### Read the Bronze Table

In [0]:
bronze_orders=spark.table("workspace.ecommerce_project.bronze_orders")
bronze_orders.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|    customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+--

### Verify the Data

In [0]:
bronze_record_count=bronze_orders.count()
print(f"Bronze record count: {bronze_record_count}")

Bronze record count: 1000


In [0]:
bronze_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_phone: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- order_amount: double (nullable = true)
 |-- final_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- delivery_date: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- shipping_state: string (nullable = true)
 |-- pincode: string (nullable = true)
 |-- ingestion_timestamp: timestamp

### Understand the Columns

In [0]:
bronze_orders.columns

['order_id',
 'customer_id',
 'customer_name',
 'customer_email',
 'customer_phone',
 'product_id',
 'product_name',
 'category',
 'quantity',
 'unit_price',
 'discount_percent',
 'order_amount',
 'final_amount',
 'payment_method',
 'payment_status',
 'order_status',
 'order_date',
 'ship_date',
 'delivery_date',
 'shipping_city',
 'shipping_state',
 'pincode',
 'ingestion_timestamp',
 'source_file',
 'batch_id',
 'source_system',
 'pipeline_id']

### Initial Data Profiling

In [0]:
from pyspark.sql import functions as F

null_summary_df=bronze_orders.select(
    *[F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in bronze_orders.columns
    ]
)
null_summary_df.show()

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+
|order_id|customer_id|customer_name|customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp|source_file|batch_id|source_system|pipeline_id|
+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+-

In [0]:
text_columns = [
    "customer_name",
    "customer_email",
    "customer_phone",
    "product_name",
    "category",
    "payment_method",
    "payment_status",
    "order_status",
    "shipping_city",
    "shipping_state",
    "pincode"
]

In [0]:
blank_summary_df=bronze_orders.select(
    *[
        F.sum((
        F.col(column).isNotNull() &
        F.trim((F.col(column))== "")
        ).cast("int")
        ).alias(column)
        for column in text_columns
    ]
)
blank_summary_df.show()

+-------------+--------------+--------------+------------+--------+--------------+--------------+------------+-------------+--------------+-------+
|customer_name|customer_email|customer_phone|product_name|category|payment_method|payment_status|order_status|shipping_city|shipping_state|pincode|
+-------------+--------------+--------------+------------+--------+--------------+--------------+------------+-------------+--------------+-------+
|            0|             0|             0|           0|       0|             0|             0|           0|            0|             0|      0|
+-------------+--------------+--------------+------------+--------+--------------+--------------+------------+-------------+--------------+-------+



### Record Our Starting Point

In [0]:
bronze_record_count=bronze_orders.count()
print(f"Starting Bronze records: {bronze_record_count}")

Starting Bronze records: 1000


### Investigate customer_name

In [0]:
display(
    bronze_orders
    .groupBy("customer_name")
    .count()
    .orderBy(F.desc("count"))
)

customer_name,count
Suresh Nair,23
Ayesha Chandran,15
Nandini Swamy,12
Ganesh Murthy,12
Yusuf Pillai,11
Vinod Chowdary,11
Rakesh Menon,11
Bhavani Devi,11
Vikram Murthy,11
Aditya Venkatesh,10


In [0]:
customer_name_issues_df=(
    bronze_orders.filter(
    F.col("customer_name").isNull() |
    (F.trim(F.col("customer_name")) == "")
    )
)
customer_name_issues_df.show()

+---------+-----------+-------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+-------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+

### Investigate customer_email

In [0]:
customer_email_issues_df=(
    bronze_orders.filter(
        F.col("customer_email").isNull() |
        F.trim((F.col("customer_email")) == "")
    )
)
customer_email_issues_df.show()

+---------+-----------+---------------+--------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|  customer_name|customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+---------------+--------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+------------

### Check Email Format

In [0]:
email_pattern=r"^[A-Za-z.%+-_]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

invalid_email_df=bronze_orders.filter(
    F.col("customer_email").isNotNull() &
    F.trim((F.col("customer_email"))!="") &
    ~F.col("customer_email").rlike(email_pattern)
)
invalid_email_df.select(
    "order_id",
    "customer_name",
    "customer_email"
).show(truncate=False)

+---------+---------------+-------------------------+
|order_id |customer_name  |customer_email           |
+---------+---------------+-------------------------+
|ORD100845|Mahesh Shetty  |                         |
|ORD100880|Revathi Iyer   |revathi iyer gmail.com   |
|ORD100795|Vinod Chowdary |vinod chowdary gmail.com |
|ORD100766|Suresh Nair    |suresh.nair.com          |
|ORD100891|Pooja Swamy    |                         |
|ORD100833|Kavya Balan    |                         |
|ORD100823|Kalpana Rajan  |                         |
|ORD100823|Kalpana Rajan  |                         |
|ORD100886|Ramya Iyer     |                         |
|ORD100858|Dinesh Naidu   |                         |
|ORD100886|Ramya Iyer     |                         |
|ORD100868|Ashok Iyer     |ashok iyer gmail.com     |
|ORD100780|Girish Naidu   |                         |
|ORD100901|Bhavani Devi   |bhavani.devi.com         |
|ORD100878|Arjun Sundaram |                         |
|ORD100857|Raghavan Sheikh|r

### Normalize the Name First

In [0]:
order_name_normalized_df=bronze_orders.withColumn(
    "customer_name",
    F.when(
        F.trim(F.col("customer_name")) == "",None
    )
    .otherwise(
        F.trim(F.col("customer_name"))
    )
)
order_name_normalized_df.show(truncate=False)

+---------+-----------+-----------------+------------------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------------+---------------+--------+-------------+-----------+
|order_id |customer_id|customer_name    |customer_email                |customer_phone|product_id|product_name       |category   |quantity|unit_price|discount_percent|order_amount|final_amount|payment_method  |payment_status|order_status|order_date|ship_date |delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp       |source_file    |batch_id|source_system|pipeline_id|
+---------+-----------+-----------------+------------------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+----------

### Normalize the Email

In [0]:
orders_name_email_normalized_df=order_name_normalized_df.withColumn(
    "customer_email",
    F.when(
        F.trim(F.col("customer_email")) == "",None
    ).otherwise(
        F.lower(F.trim(F.col("customer_email")))
    )
)
orders_name_email_normalized_df.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|    customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+--

### Derive Missing Customer Names

In [0]:
orders_name_email_normalized_df.filter(F.col("customer_name").isNull()).show(truncate=False)

+---------+-----------+-------------+------------------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------------+---------------+--------+-------------+-----------+
|order_id |customer_id|customer_name|customer_email                |customer_phone|product_id|product_name       |category   |quantity|unit_price|discount_percent|order_amount|final_amount|payment_method  |payment_status|order_status|order_date|ship_date |delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp       |source_file    |batch_id|source_system|pipeline_id|
+---------+-----------+-------------+------------------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+-------

In [0]:
order_name_email_recovered_df=(
    orders_name_email_normalized_df.withColumn(
        "customer_name",
        F.when(
            F.col("customer_name").isNull() &
            F.col("customer_email").rlike(email_pattern),
            F.initcap(
                F.regexp_replace(
                    F.split(F.col("customer_email"),"@")[0],
                    r"\.",
                    " "
                )
            )
        ).otherwise(F.col("customer_name"))
        )
)
order_name_email_recovered_df.filter(F.col("order_id")=="ORD100896").show(truncate=False)

+---------+-----------+-------------+------------------------+--------------+----------+-----------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------------+---------------+--------+-------------+-----------+
|order_id |customer_id|customer_name|customer_email          |customer_phone|product_id|product_name     |category   |quantity|unit_price|discount_percent|order_amount|final_amount|payment_method  |payment_status|order_status|order_date|ship_date |delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp       |source_file    |batch_id|source_system|pipeline_id|
+---------+-----------+-------------+------------------------+--------------+----------+-----------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+-------

### Verify the Recovery

In [0]:
display(
    order_name_email_recovered_df.filter(
        F.col("customer_name").isNotNull()
    ).select(
        "order_id",
        "customer_name",
        "customer_email"
    )
)

order_id,customer_name,customer_email
ORD100218,Zara Sundaram,zara.sundaram@yahoo.com
ORD100164,Mahesh Shetty,mahesh.shetty@rediffmail.com
ORD100064,Chitra Reddy,chitra.reddy@hotmail.com
ORD100596,Kabir Siddiqui,kabir.siddiqui@gmail.com
ORD100313,Karthik Murthy,karthik.murthy@rediffmail.com
ORD100660,Radha Chandran,radha.chandran@rediffmail.com
ORD100071,Rajesh Siddiqui,rajesh.siddiqui@rediffmail.com
ORD100724,Nandini Swamy,nandini.swamy@yahoo.com
ORD100851,Rahul Subramaniam,rahul.subramaniam@yahoo.com
ORD100463,Vikram Murthy,vikram.murthy@yahoo.com


### Identify Records That Cannot Be Recovered

In [0]:
unrecoverable_customer_df=order_name_email_recovered_df.filter(
    F.col("customer_name").isNull() &
    (
        F.col("customer_email").isNull() |
        ~F.col("customer_email").rlike(email_pattern)
    )
)
display(unrecoverable_customer_df.select(
    "order_id",
    "customer_id",
    "customer_name",
    "customer_email"
))

order_id,customer_id,customer_name,customer_email


In [0]:
invalid_email_df.show(truncate=False)

+---------+-----------+---------------+-------------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------------+---------------+--------+-------------+-----------+
|order_id |customer_id|customer_name  |customer_email           |customer_phone|product_id|product_name       |category   |quantity|unit_price|discount_percent|order_amount|final_amount|payment_method  |payment_status|order_status|order_date|ship_date |delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp       |source_file    |batch_id|source_system|pipeline_id|
+---------+-----------+---------------+-------------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+---

### Investigate customer_phone

In [0]:
bronze_orders.filter(
    F.col("customer_phone").isNull() |
    (F.trim(F.col("customer_phone")) == "")
).select(
    "order_id",
    "customer_id",
    "customer_phone"
).show(truncate=False)

+---------+-----------+--------------+
|order_id |customer_id|customer_phone|
+---------+-----------+--------------+
|ORD100937|CUST10093  |NULL          |
|ORD100811|CUST10049  |NULL          |
|ORD100903|CUST10077  |NULL          |
|ORD100861|CUST10009  |NULL          |
+---------+-----------+--------------+



In [0]:
normalized_phone_df=bronze_orders.withColumn(
    "customer_phone",
    F.regexp_replace(
        F.col("customer_phone"),
        r"^\+91[-\s]*",
        ""
        )
)

In [0]:
normalized_phone_df=normalized_phone_df.withColumn(
    "customer_phone",
    F.regexp_replace(
        F.col("customer_phone"),
        r"[^0-9]",
        ""
    )
)

In [0]:
normalized_phone_df.select(
    "order_id",
    "customer_phone"
).show(50,truncate=False)

+---------+--------------+
|order_id |customer_phone|
+---------+--------------+
|ORD100218|8249926919    |
|ORD100164|7678998028    |
|ORD100064|8113585139    |
|ORD100596|8812308209    |
|ORD100313|8327268503    |
|ORD100660|9389854268    |
|ORD100071|6698509918    |
|ORD100724|7865523129    |
|ORD100851|7608482316    |
|ORD100463|8141078352    |
|ORD100027|8469319644    |
|ORD100606|9297018781    |
|ORD100750|9599154414    |
|ORD100653|6765055833    |
|ORD100266|6903132646    |
|ORD100918|9389854268    |
|ORD100115|8185675980    |
|ORD100253|8327268503    |
|ORD100110|6575808013    |
|ORD100031|8812306873    |
|ORD100841|6345938494    |
|ORD100678|8427696198    |
|ORD100013|6190137896    |
|ORD100757|6200140141    |
|ORD100590|6513140753    |
|ORD100443|7209415691    |
|ORD100114|6191969690    |
|ORD100211|6605399561    |
|ORD100373|6191969690    |
|ORD100933|7480990401    |
|ORD100845|7678998028    |
|ORD100896|6630757561    |
|ORD100709|9601463916    |
|ORD100550|9361976301    |
|

In [0]:
normalized_phone_df.select(
    "customer_phone"
).distinct().show(100,truncate=False)

+--------------+
|customer_phone|
+--------------+
|8249926919    |
|7678998028    |
|8113585139    |
|8812308209    |
|8327268503    |
|9389854268    |
|6698509918    |
|7865523129    |
|7608482316    |
|8141078352    |
|8469319644    |
|9297018781    |
|9599154414    |
|6765055833    |
|6903132646    |
|8185675980    |
|6575808013    |
|8812306873    |
|6345938494    |
|8427696198    |
|6190137896    |
|6200140141    |
|6513140753    |
|7209415691    |
|6191969690    |
|6605399561    |
|7480990401    |
|6630757561    |
|9601463916    |
|9361976301    |
|6292587044    |
|9524806516    |
|9465341213    |
|8216070149    |
|9101815992    |
|9810303128    |
|7960514523    |
|7858511779    |
|7798086909    |
|7856528252    |
|7237859287    |
|8624636385    |
|7905817775    |
|9675770529    |
|8406002884    |
|6781057736    |
|6345824373    |
|8918739736    |
|6359154084    |
|6734036506    |
|8171069472    |
|7647099690    |
|9455281184    |
|8357109965    |
|9692362342    |
|8749431082   

In [0]:
phone_pattern=r"^[6-9][0-9]{9}$"

invalid_phone_df=normalized_phone_df.filter(
    F.col("customer_phone").isNotNull() &
    (F.trim(F.col("customer_phone"))!="") &
    ~F.col("customer_phone").rlike(phone_pattern)
)

invalid_phone_df.select(
    "order_id",
    "customer_id",
    "customer_phone"
).show(truncate=False)

+---------+-----------+--------------+
|order_id |customer_id|customer_phone|
+---------+-----------+--------------+
|ORD100773|CUST10006  |987654321     |
|ORD100777|CUST99993  |1234567890    |
|ORD100900|CUST10086  |1234567890    |
|ORD100764|CUST10038  |98765432      |
|ORD100788|CUST10109  |12345         |
|ORD100842|CUST10055  |12345         |
|ORD100876|CUST10068  |12345         |
+---------+-----------+--------------+



### Identifier Validation

In [0]:
# Check for missing identifiers
bronze_orders.select(
    F.sum((F.col("order_id").isNull()).cast("int")).alias("null_order_id"),
    F.sum((F.col("customer_id").isNull()).cast("int")).alias("null_customer_id"),
    F.sum((F.col("product_id").isNull()).cast("int")).alias("null_product_id")
).show()

+-------------+----------------+---------------+
|null_order_id|null_customer_id|null_product_id|
+-------------+----------------+---------------+
|            0|               0|              0|
+-------------+----------------+---------------+



In [0]:
bronze_orders.select(
    F.sum((F.trim(F.col("order_id"))=="").cast("int")).alias("blank_order_id"),
    F.sum((F.trim(F.col("customer_id")) == "").cast("int")).alias("blank_customer_id"),
    F.sum((F.trim(F.col("product_id"))=="").cast("int")).alias("blank_product_id")
    
).show()

+--------------+-----------------+----------------+
|blank_order_id|blank_customer_id|blank_product_id|
+--------------+-----------------+----------------+
|             0|                0|               0|
+--------------+-----------------+----------------+



### Validate order_id

In [0]:
order_id_pattern=r"^ORD[0-9]{6}$"
invalid_order_id_df=bronze_orders.filter((
    ~F.col("order_id").rlike(order_id_pattern)
))

invalid_order_id_df.select("order_id").show(truncate=False)

+--------+
|order_id|
+--------+
+--------+



### Validate customer_id

In [0]:
customer_id_pattern=r"^CUST[0-9]{5}$"
invalid_customer_id_df=bronze_orders.filter(
    ~F.col("customer_id").rlike(customer_id_pattern)
)
invalid_customer_id_df.select("customer_id").show(truncate=False)

+-----------+
|customer_id|
+-----------+
+-----------+



### Validate product_id

In [0]:
product_id_pattern=r"^PROD[0-9]{4}$"

invalid_product_id_df=bronze_orders.filter(
    ~F.col("product_id").rlike(product_id_pattern)
)

invalid_product_id_df.select("product_id").show()

+----------+
|product_id|
+----------+
+----------+



In [0]:
bronze_orders.filter(F.col("order_id")=="ORD100724").show()

+---------+-----------+-------------+--------------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|customer_name|      customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+-------------+--------------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-

### Check for Missing Product Names

In [0]:
bronze_orders.filter(
    F.col("product_name").isNull()
).select(
    "order_id",
    "product_id",
    "product_name",
    "category",
    "order_status"
).show(truncate=False)

+---------+----------+------------+-----------+------------+
|order_id |product_id|product_name|category   |order_status|
+---------+----------+------------+-----------+------------+
|ORD100792|PROD1018  |NULL        |Grocery    |Shipped     |
|ORD100881|PROD1023  |NULL        |Electronics|Delivered   |
|ORD100935|PROD1092  |NULL        |Electronics|Placed      |
|ORD100838|PROD1057  |NULL        |Home       |Confirmed   |
+---------+----------+------------+-----------+------------+



In [0]:
bronze_orders.filter(
    F.col("product_name").isNotNull() &
    (F.trim(F.col("product_name")) == "")
).select(
    "order_id",
    "product_id",
    "product_name"
).show()

+--------+----------+------------+
|order_id|product_id|product_name|
+--------+----------+------------+
+--------+----------+------------+



In [0]:
bronze_orders.filter(F.col("product_id")=="PROD1023").select("product_id","product_name").show()

+----------+-------------------+
|product_id|       product_name|
+----------+-------------------+
|  PROD1023|Wireless Headphones|
|  PROD1023|Wireless Headphones|
|  PROD1023|Wireless Headphones|
|  PROD1023|Wireless Headphones|
|  PROD1023|               NULL|
|  PROD1023|Wireless Headphones|
|  PROD1023|Wireless Headphones|
|  PROD1023|Wireless Headphones|
|  PROD1023|Wireless Headphones|
|  PROD1023|Wireless Headphones|
|  PROD1023|Wireless Headphones|
|  PROD1023|Wireless Headphones|
|  PROD1023|Wireless Headphones|
|  PROD1023|Wireless Headphones|
|  PROD1023|Wireless Headphones|
+----------+-------------------+



In [0]:
bronze_orders.groupBy(
    "product_id"
    ).agg(F.countDistinct(
        F.trim(F.col("product_id"))).alias("distinct_product_id")).orderBy("product_id").show()

+----------+-------------------+
|product_id|distinct_product_id|
+----------+-------------------+
|  PROD1001|                  1|
|  PROD1002|                  1|
|  PROD1003|                  1|
|  PROD1004|                  1|
|  PROD1005|                  1|
|  PROD1006|                  1|
|  PROD1007|                  1|
|  PROD1008|                  1|
|  PROD1009|                  1|
|  PROD1010|                  1|
|  PROD1011|                  1|
|  PROD1012|                  1|
|  PROD1013|                  1|
|  PROD1014|                  1|
|  PROD1015|                  1|
|  PROD1016|                  1|
|  PROD1017|                  1|
|  PROD1018|                  1|
|  PROD1019|                  1|
|  PROD1020|                  1|
+----------+-------------------+
only showing top 20 rows


### Create a Product Reference

In [0]:
product_reference = (
    bronze_orders.filter(
        F.col("product_id").isNotNull() &
        (F.trim(F.col("product_name"))!="")
    )
    .select(
        "product_id",
        F.trim(F.col("product_name")).alias("reference_product_name")
    ).dropDuplicates(["product_id"])
)

In [0]:
product_reference.show(product_reference.count())

+----------+----------------------+
|product_id|reference_product_name|
+----------+----------------------+
|  PROD1064|           Cricket Bat|
|  PROD1073|                 Mouse|
|  PROD1066|                 Books|
|  PROD1036|          Water Bottle|
|  PROD1075|          Cotton Shirt|
|  PROD1048|     Bluetooth Speaker|
|  PROD1100|         Mixer Grinder|
|  PROD1020|           Cricket Bat|
|  PROD1096|       Laptop Backpack|
|  PROD1049|           LED Monitor|
|  PROD1006|              Keyboard|
|  PROD1081|      Herbal Face Wash|
|  PROD1007|                 Mouse|
|  PROD1018|   Assorted Spices Box|
|  PROD1053|          Cotton Shirt|
|  PROD1031|          Cotton Shirt|
|  PROD1005|           LED Monitor|
|  PROD1026|     Bluetooth Speaker|
|  PROD1098|         Running Shoes|
|  PROD1074|       Laptop Backpack|
|  PROD1041|              Yoga Mat|
|  PROD1034|         Mixer Grinder|
|  PROD1067|   Wireless Headphones|
|  PROD1092|     Bluetooth Speaker|
|  PROD1069|           Smart

### Join the Reference Back to the Orders

In [0]:
orders_with_product_reference = (
    bronze_orders.join(
    product_reference,
    on="product_id",
    how="left"
)
)

In [0]:
orders_with_product_reference.filter(F.col("product_name").isNull()).show()

+----------+---------+-----------+-------------+--------------------+--------------+------------+-----------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+----------------------+
|product_id| order_id|customer_id|customer_name|      customer_email|customer_phone|product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|reference_product_name|
+----------+---------+-----------+-------------+--------------------+--------------+------------+-----------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+------

In [0]:
orders_with_product_name = (
    orders_with_product_reference.withColumn(
        "product_name",
        F.when(
            F.col("product_name").isNull() |
            (F.trim(F.col("product_name"))==""),
            F.col("reference_product_name")
        ).otherwise(
            F.trim(F.col("product_name"))
        )
    ).drop("reference_product_name")
)
orders_with_product_name.show()

+----------+---------+-----------+-----------------+--------------------+--------------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
|product_id| order_id|customer_id|    customer_name|      customer_email|customer_phone|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+----------+---------+-----------+-----------------+--------------------+--------------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+--

### Verify the Result

In [0]:
orders_with_product_name.filter(
    F.col("product_name").isNull() |
    (F.trim(F.col("product_name"))== "")
).select(
    "order_id",
    "product_id",
    "product_name"
).show()

+--------+----------+------------+
|order_id|product_id|product_name|
+--------+----------+------------+
+--------+----------+------------+



In [0]:
orders_with_product_name.filter(F.col("product_id")=="PROD1092").show()

+----------+---------+-----------+----------------+--------------------+--------------+-----------------+------------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
|product_id| order_id|customer_id|   customer_name|      customer_email|customer_phone|     product_name|    category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+----------+---------+-----------+----------------+--------------------+--------------+-----------------+------------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+--------

In [0]:
category_null=bronze_orders.select(
    F.sum((F.col("category").isNull()).cast("int")).alias("null_category")
)
category_null.show()

+-------------+
|null_category|
+-------------+
|            0|
+-------------+



### Discover the Existing Categories

In [0]:
bronze_orders.select("category").distinct().show()

+------------+
|    category|
+------------+
|      Sports|
| Electronics|
|       Books|
|        Home|
|     Fashion|
|      Beauty|
|     Grocery|
|            |
|     grocery|
|            |
|  Electronic|
|      Fashon|
| Electronics|
|      sports|
| electronics|
|ELECTRONICS |
|     Beauty |
+------------+



### Analyze Category Frequency

In [0]:
bronze_orders.groupBy("category").count().orderBy(F.desc("count")).show(truncate=False)


+------------+-----+
|category    |count|
+------------+-----+
|Electronics |394  |
|Home        |143  |
|Fashion     |133  |
|Beauty      |83   |
|Books       |80   |
|Grocery     |76   |
|Sports      |67   |
|            |10   |
|            |4    |
| Electronics|2    |
|Electronic  |2    |
|Fashon      |1    |
|sports      |1    |
|grocery     |1    |
|electronics |1    |
|ELECTRONICS |1    |
|Beauty      |1    |
+------------+-----+



### Check for Missing or Blank Categories

In [0]:
bronze_orders.filter(
    F.col("category").isNull() |
    (F.trim(F.col("category"))=="")
).select("order_id","product_id","product_name","category").show(truncate=False)

+---------+----------+-------------------+--------+
|order_id |product_id|product_name       |category|
+---------+----------+-------------------+--------+
|ORD100849|PROD1084  |Assorted Spices Box|        |
|ORD100810|PROD1059  |Herbal Face Wash   |        |
|ORD100946|PROD1010  |Running Shoes      |        |
|ORD100854|PROD1074  |Laptop Backpack    |        |
|ORD100913|PROD1004  |Bluetooth Speaker  |        |
|ORD100772|PROD1083  |Organic Rice Pack  |        |
|ORD100798|PROD1041  |Yoga Mat           |        |
|ORD100771|PROD1016  |Hair Serum         |        |
|ORD100913|PROD1004  |Bluetooth Speaker  |        |
|ORD100948|PROD1039  |Organic Rice Pack  |        |
|ORD100791|PROD1053  |Cotton Shirt       |        |
|ORD100825|PROD1093  |LED Monitor        |        |
|ORD100903|PROD1052  |Laptop Backpack    |        |
|ORD100816|PROD1015  |Herbal Face Wash   |        |
+---------+----------+-------------------+--------+



### Normalize the Category for Investigation

In [0]:
category_analysis=(
    bronze_orders.withColumn(
        "normalized_category",
        F.upper(F.trim(F.col("category")))
    )
)
category_analysis.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+-------------------+
| order_id|customer_id|    customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|normalized_category|
+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+------------

In [0]:
category_analysis.groupBy("normalized_category").count().orderBy(F.desc("count")).show()

+-------------------+-----+
|normalized_category|count|
+-------------------+-----+
|        ELECTRONICS|  398|
|               HOME|  143|
|            FASHION|  133|
|             BEAUTY|   84|
|              BOOKS|   80|
|            GROCERY|   77|
|             SPORTS|   68|
|                   |   14|
|         ELECTRONIC|    2|
|             FASHON|    1|
+-------------------+-----+



### Standardize Valid Variations

In [0]:
orders_with_category=bronze_orders.withColumn(
    "category",
    F.upper(F.trim(F.col("category")))
)
orders_with_category.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|    customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+--

In [0]:
valid_category=[
    "ELECTRONICS",
    "HOME",
    "FASHION",
    "BEAUTY",
    "BOOKS",
    "GROCERY",
    "SPORTS"
]

In [0]:
invalid_category_df=(
    orders_with_category.filter(
        F.col("category").isNotNull() &
        ~F.col("category").isin(valid_category)
    )
)
invalid_category_df.select(
    "order_id",
    "product_id",
    "product_name",
    "category"
).show()

+---------+----------+-------------------+----------+
| order_id|product_id|       product_name|  category|
+---------+----------+-------------------+----------+
|ORD100849|  PROD1084|Assorted Spices Box|          |
|ORD100810|  PROD1059|   Herbal Face Wash|          |
|ORD100946|  PROD1010|      Running Shoes|          |
|ORD100854|  PROD1074|    Laptop Backpack|          |
|ORD100913|  PROD1004|  Bluetooth Speaker|          |
|ORD100813|  PROD1091|        Smart Watch|ELECTRONIC|
|ORD100772|  PROD1083|  Organic Rice Pack|          |
|ORD100798|  PROD1041|           Yoga Mat|          |
|ORD100775|  PROD1033|           Backpack|    FASHON|
|ORD100771|  PROD1016|         Hair Serum|          |
|ORD100813|  PROD1091|        Smart Watch|ELECTRONIC|
|ORD100913|  PROD1004|  Bluetooth Speaker|          |
|ORD100948|  PROD1039|  Organic Rice Pack|          |
|ORD100791|  PROD1053|       Cotton Shirt|          |
|ORD100825|  PROD1093|        LED Monitor|          |
|ORD100903|  PROD1052|    La

### Investigate the Quantity Column

In [0]:
bronze_orders.select(
    "order_id",
    "product_id",
    "quantity"
).show(20)

+---------+----------+--------+
| order_id|product_id|quantity|
+---------+----------+--------+
|ORD100218|  PROD1064|       8|
|ORD100164|  PROD1073|       4|
|ORD100064|  PROD1066|       4|
|ORD100596|  PROD1036|       9|
|ORD100313|  PROD1075|      10|
|ORD100660|  PROD1048|       5|
|ORD100071|  PROD1100|       6|
|ORD100724|  PROD1020|       6|
|ORD100851|  PROD1096|      10|
|ORD100463|  PROD1049|       9|
|ORD100027|  PROD1006|       6|
|ORD100606|  PROD1081|       8|
|ORD100750|  PROD1007|       4|
|ORD100653|  PROD1018|       7|
|ORD100266|  PROD1053|       8|
|ORD100918|  PROD1031|       8|
|ORD100115|  PROD1005|       3|
|ORD100253|  PROD1026|       2|
|ORD100110|  PROD1098|       3|
|ORD100031|  PROD1074|       5|
+---------+----------+--------+
only showing top 20 rows


In [0]:
bronze_orders.filter(F.col("quantity").isNull()).select(
    "order_id",
    "product_id",
    "quantity"
).show()

+---------+----------+--------+
| order_id|product_id|quantity|
+---------+----------+--------+
|ORD100881|  PROD1023|    NULL|
|ORD100892|  PROD1086|    NULL|
|ORD100900|  PROD1077|    NULL|
|ORD100873|  PROD1012|    NULL|
|ORD100910|  PROD1064|    NULL|
|ORD100768|  PROD1028|    NULL|
+---------+----------+--------+



In [0]:
invalid_quantity_df=bronze_orders.filter(
    F.col("quantity").isNull() |
    (F.col("quantity")<=0)
)
invalid_quantity_df.show()

+---------+-----------+---------------+--------------------+--------------+----------+-----------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|  customer_name|      customer_email|customer_phone|product_id|     product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+---------------+--------------------+--------------+----------+-----------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+

### Profile the Quantity Distribution

In [0]:
bronze_orders.select(
    F.min("quantity").alias("min quantity"),
    F.max("quantity").alias("max quantity"),
    F.avg("quantity").alias("average quantity")
).show()

+------------+------------+----------------+
|min quantity|max quantity|average quantity|
+------------+------------+----------------+
|          -5|          11|5.49195171026157|
+------------+------------+----------------+



In [0]:
quantity_quality_summary=bronze_orders.select(
    F.sum((F.col("quantity").isNull()).cast("int")).alias("Null quantity"),
    F.sum((F.col("quantity")==0).cast("int")).alias("Zero quantity"),
    F.sum((F.col("quantity")<0).cast("int")).alias("Negative quantity"),
    F.sum((F.col("quantity")>0).cast("int")).alias("Positive quantity")
)
quantity_quality_summary.show()

+-------------+-------------+-----------------+-----------------+
|Null quantity|Zero quantity|Negative quantity|Positive quantity|
+-------------+-------------+-----------------+-----------------+
|            6|            5|                5|              984|
+-------------+-------------+-----------------+-----------------+



### Investigate the Column

In [0]:
bronze_orders.select(
    "order_id",
    "product_id",
    "product_name",
    "quantity",
    "unit_price"
).show()

+---------+----------+-------------------+--------+----------+
| order_id|product_id|       product_name|quantity|unit_price|
+---------+----------+-------------------+--------+----------+
|ORD100218|  PROD1064|        Cricket Bat|       8|   5037.33|
|ORD100164|  PROD1073|              Mouse|       4|   2091.76|
|ORD100064|  PROD1066|              Books|       4|   2731.88|
|ORD100596|  PROD1036|       Water Bottle|       9|    334.78|
|ORD100313|  PROD1075|       Cotton Shirt|      10|   2379.61|
|ORD100660|  PROD1048|  Bluetooth Speaker|       5|   7784.98|
|ORD100071|  PROD1100|      Mixer Grinder|       6|   8724.76|
|ORD100724|  PROD1020|        Cricket Bat|       6|   5502.35|
|ORD100851|  PROD1096|    Laptop Backpack|      10|   4238.02|
|ORD100463|  PROD1049|        LED Monitor|       9|  38853.83|
|ORD100027|  PROD1006|           Keyboard|       6|    4858.4|
|ORD100606|  PROD1081|   Herbal Face Wash|       8|     737.0|
|ORD100750|  PROD1007|              Mouse|       4|   2

In [0]:
bronze_orders.filter(F.col("unit_price").isNull()).select(
    "order_id",
    "product_id",
    "product_name",
    "quantity",
    "unit_price"
).show()

+---------+----------+-------------------+--------+----------+
| order_id|product_id|       product_name|quantity|unit_price|
+---------+----------+-------------------+--------+----------+
|ORD100845|  PROD1090|         Smartphone|       7|      NULL|
|ORD100936|  PROD1045|Wireless Headphones|       2|      NULL|
|ORD100827|  PROD1035|       Office Chair|       2|      NULL|
|ORD100944|  PROD1034|      Mixer Grinder|       9|      NULL|
|ORD100910|  PROD1064|        Cricket Bat|    NULL|      NULL|
|ORD100914|  PROD9992|        LED Monitor|      10|      NULL|
+---------+----------+-------------------+--------+----------+



In [0]:
bronze_orders.filter(F.col("unit_price")<=0).select(
    "order_id",
    "product_id",
    "product_name",
    "quantity",
    "unit_price"
).show()

+---------+----------+-----------------+--------+----------+
| order_id|product_id|     product_name|quantity|unit_price|
+---------+----------+-----------------+--------+----------+
|ORD100819|  PROD1042|      Cricket Bat|       3|       0.0|
|ORD100917|  PROD1061|Organic Rice Pack|       5|   -2500.0|
|ORD100792|  PROD1018|             NULL|       6|       0.0|
|ORD100819|  PROD1042|      Cricket Bat|       3|       0.0|
|ORD100789|  PROD1051|            Mouse|       4|    -100.0|
|ORD100907|  PROD9992|Fiction Novel Set|       9|    -100.0|
|ORD100887|  PROD1095|            Mouse|       6|   -2500.0|
|ORD100800|  PROD1017|Organic Rice Pack|      10|    -100.0|
+---------+----------+-----------------+--------+----------+



In [0]:
invalid_unit_price_df=bronze_orders.filter(
    (F.col("unit_price").isNull()) |
    (F.col("unit_price")<=0)
)
invalid_unit_price_df.show()

+---------+-----------+----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|   customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-----

### Profile the Price Values

In [0]:
bronze_orders.select(
    F.min("unit_price").alias("Minimum unit price"),
    F.max("unit_price").alias("Maximum unit price"),
    F.avg("unit_price").alias("Average unit price")
).show()

+------------------+------------------+------------------+
|Minimum unit price|Maximum unit price|Average unit price|
+------------------+------------------+------------------+
|           -2500.0|         148046.97|  9459.41829979878|
+------------------+------------------+------------------+



In [0]:
bronze_orders.select(
    F.sum((F.col("unit_price").isNull()).cast("int")).alias("Null unit price"),
    F.sum((F.col("unit_price")==0).cast("int")).alias("Zero unit price"),
    F.sum((F.col("unit_price")>0).cast("int")).alias("Postive unit price"),
    F.sum((F.col("unit_price")<0).cast("int")).alias("Negative unit price")
).show()

+---------------+---------------+------------------+-------------------+
|Null unit price|Zero unit price|Postive unit price|Negative unit price|
+---------------+---------------+------------------+-------------------+
|              6|              3|               986|                  5|
+---------------+---------------+------------------+-------------------+



### Investigate the Column

In [0]:
bronze_orders.select(
    "order_id",
    "quantity",
    "unit_price",
    "discount_percent"
).show()

+---------+--------+----------+----------------+
| order_id|quantity|unit_price|discount_percent|
+---------+--------+----------+----------------+
|ORD100218|       8|   5037.33|               0|
|ORD100164|       4|   2091.76|              80|
|ORD100064|       4|   2731.88|              10|
|ORD100596|       9|    334.78|              80|
|ORD100313|      10|   2379.61|              50|
|ORD100660|       5|   7784.98|              10|
|ORD100071|       6|   8724.76|               0|
|ORD100724|       6|   5502.35|              60|
|ORD100851|      10|   4238.02|               0|
|ORD100463|       9|  38853.83|              30|
|ORD100027|       6|    4858.4|              10|
|ORD100606|       8|     737.0|               0|
|ORD100750|       4|   2854.72|              50|
|ORD100653|       7|    584.08|              15|
|ORD100266|       8|   1994.94|              80|
|ORD100918|       8|    553.11|              30|
|ORD100115|       3|  33830.08|              60|
|ORD100253|       2|

In [0]:
bronze_orders.select("discount_percent").distinct().show()

+----------------+
|discount_percent|
+----------------+
|               0|
|              80|
|              10|
|              50|
|              60|
|              30|
|              15|
|              25|
|              20|
|               5|
|              40|
|             120|
|             150|
|             -10|
|             100|
+----------------+



In [0]:
bronze_orders.filter(
    F.col("discount_percent").isNull()
    ).select(
        "order_id",
        "product_id",
        "discount_percent"
    ).show()

+--------+----------+----------------+
|order_id|product_id|discount_percent|
+--------+----------+----------------+
+--------+----------+----------------+



### Identify Invalid Discounts

In [0]:
invalid_discount_df=(
    bronze_orders.filter(
        F.col("discount_percent").isNotNull() & (
            (F.col("discount_percent")<0) |
            (F.col("discount_percent")>80)
        )
    )
)
invalid_discount_df.show()

+---------+-----------+-----------------+--------------------+--------------+----------+----------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|    customer_name|      customer_email|customer_phone|product_id|    product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+-----------------+--------------------+--------------+----------+----------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-----------

### Inspect the Existing Values

In [0]:
bronze_orders.select(
    "order_id",
    "quantity",
    "unit_price",
    "order_amount"
).show()

+---------+--------+----------+------------+
| order_id|quantity|unit_price|order_amount|
+---------+--------+----------+------------+
|ORD100218|       8|   5037.33|    40298.64|
|ORD100164|       4|   2091.76|     8367.04|
|ORD100064|       4|   2731.88|    10927.52|
|ORD100596|       9|    334.78|     3013.02|
|ORD100313|      10|   2379.61|     23796.1|
|ORD100660|       5|   7784.98|     38924.9|
|ORD100071|       6|   8724.76|    52348.56|
|ORD100724|       6|   5502.35|     33014.1|
|ORD100851|      10|   4238.02|     42380.2|
|ORD100463|       9|  38853.83|   349684.47|
|ORD100027|       6|    4858.4|     29150.4|
|ORD100606|       8|     737.0|      5896.0|
|ORD100750|       4|   2854.72|    11418.88|
|ORD100653|       7|    584.08|     4088.56|
|ORD100266|       8|   1994.94|    15959.52|
|ORD100918|       8|    553.11|     4424.88|
|ORD100115|       3|  33830.08|   101490.24|
|ORD100253|       2|    990.81|     1981.62|
|ORD100110|       3|   4588.21|    13764.63|
|ORD100031

### Calculate the Expected Order Amount

In [0]:
order_amount_check_df=bronze_orders.withColumn(
    "expected_order_amount",
    F.round(
        F.col("quantity")*F.col("unit_price"),
        2
        )
)
order_amount_check_df.select(
    "order_id",
    "quantity",
    "unit_price",
    "order_amount",
    "expected_order_amount",
    ).show()

+---------+--------+----------+------------+---------------------+
| order_id|quantity|unit_price|order_amount|expected_order_amount|
+---------+--------+----------+------------+---------------------+
|ORD100218|       8|   5037.33|    40298.64|             40298.64|
|ORD100164|       4|   2091.76|     8367.04|              8367.04|
|ORD100064|       4|   2731.88|    10927.52|             10927.52|
|ORD100596|       9|    334.78|     3013.02|              3013.02|
|ORD100313|      10|   2379.61|     23796.1|              23796.1|
|ORD100660|       5|   7784.98|     38924.9|              38924.9|
|ORD100071|       6|   8724.76|    52348.56|             52348.56|
|ORD100724|       6|   5502.35|     33014.1|              33014.1|
|ORD100851|      10|   4238.02|     42380.2|              42380.2|
|ORD100463|       9|  38853.83|   349684.47|            349684.47|
|ORD100027|       6|    4858.4|     29150.4|              29150.4|
|ORD100606|       8|     737.0|      5896.0|               589

In [0]:
# why cross-column check we are doing beacuse see the unit price and quantity with order amount and expected order amount
invalid_order_amount_df=order_amount_check_df.filter(
    F.col("order_amount").isNull() |
    F.col("expected_order_amount").isNull() 
)
invalid_order_amount_df.show()

+---------+-----------+----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+---------------------+
| order_id|customer_id|   customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|expected_order_amount|
+---------+-----------+----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+-----------

In [0]:
invalid_order_amount_df=order_amount_check_df.filter(
    F.col("order_amount").isNull() |
    F.col("expected_order_amount").isNull() |
    (
    F.abs(
        F.col("order_amount")-F.col("expected_order_amount")
    )>0.01
    )
)
invalid_order_amount_df.select(
    "order_id",
    "quantity",
    "unit_price",
    "order_amount",
    "expected_order_amount"
).show()

+---------+--------+----------+------------+---------------------+
| order_id|quantity|unit_price|order_amount|expected_order_amount|
+---------+--------+----------+------------+---------------------+
|ORD100845|       7|      NULL|   408564.24|                 NULL|
|ORD100819|       3|       0.0|     7119.51|                  0.0|
|ORD100934|       0|   1428.85|     4286.55|                  0.0|
|ORD100849|       8|    584.76|     4383.84|              4678.08|
|ORD100874|       0|   1477.02|     14770.2|                  0.0|
|ORD100936|       2|      NULL|      2715.7|                 NULL|
|ORD100922|      -1|   5290.23|     5290.23|             -5290.23|
|ORD100755|      -5|    690.66|     5525.28|              -3453.3|
|ORD100853|      -1|  10109.03|    50545.15|            -10109.03|
|ORD100917|       5|   -2500.0|     6399.25|             -12500.0|
|ORD100792|       6|       0.0|      5606.7|                  0.0|
|ORD100806|      -5|    433.96|     1214.42|              -216

### Inspect the Existing Values

In [0]:
bronze_orders.select(
    "order_id",
    "order_amount",
    "discount_percent",
    "final_amount"
).show()

+---------+------------+----------------+------------+
| order_id|order_amount|discount_percent|final_amount|
+---------+------------+----------------+------------+
|ORD100218|    40298.64|               0|    40298.64|
|ORD100164|     8367.04|              80|     1673.41|
|ORD100064|    10927.52|              10|     9834.77|
|ORD100596|     3013.02|              80|       602.6|
|ORD100313|     23796.1|              50|    11898.05|
|ORD100660|     38924.9|              10|    35032.41|
|ORD100071|    52348.56|               0|    52348.56|
|ORD100724|     33014.1|              60|    13205.64|
|ORD100851|     42380.2|               0|     42380.2|
|ORD100463|   349684.47|              30|   244779.13|
|ORD100027|     29150.4|              10|    26235.36|
|ORD100606|      5896.0|               0|      5896.0|
|ORD100750|    11418.88|              50|     5709.44|
|ORD100653|     4088.56|              15|     3475.28|
|ORD100266|    15959.52|              80|      3191.9|
|ORD100918

### Calculate the Expected Final Amount

In [0]:
final_amount_check_df=bronze_orders.withColumn(
    "expected_final_amount",
    F.round(
        (F.col("order_amount")-(F.col("order_amount")*(F.col("discount_percent")/100)))
        ,2
        )
)
final_amount_check_df.select(
    "order_id",
    "order_amount",
    "discount_percent",
    "final_amount",
    "expected_final_amount"
    ).show()

+---------+------------+----------------+------------+---------------------+
| order_id|order_amount|discount_percent|final_amount|expected_final_amount|
+---------+------------+----------------+------------+---------------------+
|ORD100218|    40298.64|               0|    40298.64|             40298.64|
|ORD100164|     8367.04|              80|     1673.41|              1673.41|
|ORD100064|    10927.52|              10|     9834.77|              9834.77|
|ORD100596|     3013.02|              80|       602.6|                602.6|
|ORD100313|     23796.1|              50|    11898.05|             11898.05|
|ORD100660|     38924.9|              10|    35032.41|             35032.41|
|ORD100071|    52348.56|               0|    52348.56|             52348.56|
|ORD100724|     33014.1|              60|    13205.64|             13205.64|
|ORD100851|     42380.2|               0|     42380.2|              42380.2|
|ORD100463|   349684.47|              30|   244779.13|            244779.13|

### Identify Mismatches

In [0]:
invalid_final_amount_df=final_amount_check_df.filter(
    F.col("final_amount").isNull() |
    F.col("expected_final_amount").isNull() |
    (
        F.abs(
            F.col("final_amount")-F.col("expected_final_amount")
        )>0.01
    )
)
invalid_final_amount_df.select(
    "order_id",
    "order_amount",
    "discount_percent",
    "final_amount",
    "expected_final_amount"
).show()

+---------+------------+----------------+------------+---------------------+
| order_id|order_amount|discount_percent|final_amount|expected_final_amount|
+---------+------------+----------------+------------+---------------------+
|ORD100443|   241208.31|               0|   241278.64|            241208.31|
|ORD100756|     26443.3|              50|    13345.12|             13221.65|
|ORD100849|     4383.84|              10|     4210.27|              3945.46|
|ORD100713|     26746.3|               5|    25408.98|             25408.99|
|ORD100801|    19312.76|             120|    19312.76|             -3862.55|
|ORD100350|     3849.75|              50|     1924.87|              1924.88|
|ORD100245|     5565.75|              50|     2782.87|              2782.88|
|ORD100777|    63684.12|             150|    63684.12|            -31842.06|
|ORD100388|    74932.38|              25|    56199.28|             56199.29|
|ORD100806|     1214.42|              60|      520.75|               485.77|

### Discover Payment Methods

In [0]:
bronze_orders.select("payment_method").distinct().show(truncate=False)

+----------------+
|payment_method  |
+----------------+
|Net Banking     |
|Debit Card      |
|UPI             |
|Wallet          |
|Cash on Delivery|
|Credit Card     |
|DebitCard       |
|wallet          |
|NetBanking      |
| UPI            |
|debit card      |
|cash on delivery|
|Credit card     |
+----------------+



### Group and Count Each Payment Method

In [0]:
bronze_orders.groupBy("payment_method").count().orderBy("count",ascending=False).show()

+----------------+-----+
|  payment_method|count|
+----------------+-----+
|          Wallet|  177|
|      Debit Card|  176|
|Cash on Delivery|  170|
|     Credit Card|  164|
|     Net Banking|  157|
|             UPI|  148|
|     Credit card|    2|
|      debit card|    1|
|          wallet|    1|
|      NetBanking|    1|
|       DebitCard|    1|
|             UPI|    1|
|cash on delivery|    1|
+----------------+-----+



### Normalize Whitespace and Case

In [0]:
payment_method_analysis=bronze_orders.withColumn(
    "normalized_payment_method",
    F.upper(
        F.trim(
        F.col("payment_method")
    )
    )
)
payment_method_analysis.groupBy("normalized_payment_method").count().orderBy(F.desc("count")).show()

+-------------------------+-----+
|normalized_payment_method|count|
+-------------------------+-----+
|                   WALLET|  178|
|               DEBIT CARD|  177|
|         CASH ON DELIVERY|  171|
|              CREDIT CARD|  166|
|              NET BANKING|  157|
|                      UPI|  149|
|                DEBITCARD|    1|
|               NETBANKING|    1|
+-------------------------+-----+



### Apply the Standardization

In [0]:
payment_method_mapping={
    'UPI':'UPI',
    'DEBIT CARD':'DEBIT CARD',
    'DEBITCARD':'DEBIT CARD',
    'CREDIT CARD':'CREDIT CARD',
    'CREDITCARD':'CREDIT CARD',
    'COD':'CASH ON DELIVERY',
    'CASH ON DELIVERY':'CASH ON DELIVERY',
    'NET BANKING':'NET BANKING',
    'NETBANKING':'NET BANKING',
    'WALLET':'WALLET'
}

In [0]:
order_with_payment_method=payment_method_analysis.withColumn(
    "payment_method",
        F.create_map(
            *[
                F.lit(x)
           for pair in payment_method_mapping.items()
           for x in pair 
            ]
        )[F.col("normalized_payment_method")]
)
order_with_payment_method.select(
    "payment_method"
).show()

+----------------+
|  payment_method|
+----------------+
|     NET BANKING|
|     NET BANKING|
|      DEBIT CARD|
|             UPI|
|     NET BANKING|
|             UPI|
|          WALLET|
|CASH ON DELIVERY|
|          WALLET|
|CASH ON DELIVERY|
|      DEBIT CARD|
|             UPI|
|          WALLET|
|     NET BANKING|
|      DEBIT CARD|
|CASH ON DELIVERY|
|     NET BANKING|
|          WALLET|
|     NET BANKING|
|             UPI|
+----------------+
only showing top 20 rows


### Discover the Existing Values

In [0]:
bronze_orders.select("payment_status").distinct().show(truncate=False)

+--------------+
|payment_status|
+--------------+
|Paid          |
|Refunded      |
|Pending       |
|Failed        |
|paid          |
|Refundd       |
|pending       |
|Success       |
|PAID          |
+--------------+



In [0]:
bronze_orders.groupBy("payment_status").count().orderBy(F.desc("count")).show()

+--------------+-----+
|payment_status|count|
+--------------+-----+
|          Paid|  704|
|      Refunded|  170|
|       Pending|   91|
|        Failed|   27|
|       Success|    4|
|          paid|    1|
|       Refundd|    1|
|       pending|    1|
|          PAID|    1|
+--------------+-----+



In [0]:
payment_status_analysis=bronze_orders.withColumn(
    "normalized_payment_status",
    F.upper(
        F.trim(
            F.col("payment_status")
            )
    )
)

In [0]:
payment_status_analysis.groupBy("normalized_payment_status").count().orderBy("count",ascending=False).show()

+-------------------------+-----+
|normalized_payment_status|count|
+-------------------------+-----+
|                     PAID|  706|
|                 REFUNDED|  170|
|                  PENDING|   92|
|                   FAILED|   27|
|                  SUCCESS|    4|
|                  REFUNDD|    1|
+-------------------------+-----+



In [0]:
payment_status_cleaned=payment_status_analysis.withColumn(
    "payment_status",
    F.when(
        F.col("normalized_payment_status")=="SUCCESS",
        F.lit("PAID")
    ).when(
        F.col("normalized_payment_status") == "REFUNDD",
        F.lit("REFUNDED")
    )
    .otherwise(F.col("normalized_payment_status"))
)
payment_status_cleaned.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+-------------------------+
| order_id|customer_id|    customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|normalized_payment_status|
+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+

### Verify the Standardization

In [0]:
payment_status_cleaned.groupBy("payment_status").count().orderBy(F.desc("count")).show()

+--------------+-----+
|payment_status|count|
+--------------+-----+
|          PAID|  710|
|      REFUNDED|  171|
|       PENDING|   92|
|        FAILED|   27|
+--------------+-----+



### Discover the Existing Values

In [0]:
bronze_orders.select(
    "order_status"
).distinct().show()

+------------+
|order_status|
+------------+
|     Shipped|
|   Cancelled|
|   Delivered|
|    Returned|
|      Placed|
|   Confirmed|
|      placed|
|      shiped|
|  Delivered |
|    deliverd|
|       Placd|
+------------+



In [0]:
bronze_orders.groupBy("order_status").count().orderBy(F.desc("count")).show()

+------------+-----+
|order_status|count|
+------------+-----+
|   Delivered|  563|
|    Returned|  104|
|   Cancelled|   96|
|     Shipped|   92|
|   Confirmed|   77|
|      Placed|   61|
|    deliverd|    2|
|  Delivered |    2|
|      placed|    1|
|      shiped|    1|
|       Placd|    1|
+------------+-----+



### Normalize Whitespace and Case

In [0]:
order_status_analysis=bronze_orders.withColumn(
    "normalized_order_status",
    F.initcap(
        F.trim(
            F.col("order_status")
        )
    )
)

In [0]:
order_status_analysis.groupBy("normalized_order_status").count().orderBy("count",ascending=False).show()

+-----------------------+-----+
|normalized_order_status|count|
+-----------------------+-----+
|              Delivered|  565|
|               Returned|  104|
|              Cancelled|   96|
|                Shipped|   92|
|              Confirmed|   77|
|                 Placed|   62|
|               Deliverd|    2|
|                 Shiped|    1|
|                  Placd|    1|
+-----------------------+-----+



In [0]:
known_order_status=[
    'Delivered',
    'Shipped',
    'Placed',
    'Cancelled',
    'Returned',
    'Confirmed',
    'Deliverd',
    'Shiped',
    'Placd'
]
invalid_order_status_df=order_status_analysis.filter(
    ~F.col("normalized_order_status").isin(known_order_status)
)
invalid_order_status_df.show()

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+-----------------------+
|order_id|customer_id|customer_name|customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp|source_file|batch_id|source_system|pipeline_id|normalized_order_status|
+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------

In [0]:
order_status_cleaned=order_status_analysis.withColumn(
    "order_status",
    F.when(
        F.col("normalized_order_status")=="Deliverd",
        F.lit("Delivered")
    )
    .when(
        F.col("normalized_order_status")=="Shiped",
        F.lit("Shipped")
    )
    .when(
        F.col("normalized_order_status")=="Placd",
        F.lit("Placed")
    )
    .otherwise(F.col("normalized_order_status"))
)

In [0]:
order_status_cleaned.groupBy("order_status").count().orderBy(F.desc("count")).show()

+------------+-----+
|order_status|count|
+------------+-----+
|   Delivered|  567|
|    Returned|  104|
|   Cancelled|   96|
|     Shipped|   93|
|   Confirmed|   77|
|      Placed|   63|
+------------+-----+



### Step 15 — Date Validation and Parsing

### First Understand the Raw Values

In [0]:
bronze_orders.select(
    "order_id",
    "order_date",
    "ship_date",
    "delivery_date"
).show(truncate=False)

+---------+----------+----------+-------------+
|order_id |order_date|ship_date |delivery_date|
+---------+----------+----------+-------------+
|ORD100218|2025-10-18|2025-10-20|NULL         |
|ORD100164|2025-12-13|NULL      |NULL         |
|ORD100064|2025-11-27|NULL      |NULL         |
|ORD100596|2025-09-06|2025-09-07|2025-09-08   |
|ORD100313|2025-12-12|2025-12-15|2025-12-19   |
|ORD100660|2025-01-25|2025-01-26|2025-01-30   |
|ORD100071|2025-11-28|2025-12-01|2025-12-02   |
|ORD100724|2025-11-08|2025-11-12|2025-11-16   |
|ORD100851|2025-08-16|NULL      |2025-08-23   |
|ORD100463|2025-07-19|2025-07-20|NULL         |
|ORD100027|2025-07-11|2025-07-13|2025-07-18   |
|ORD100606|2025-07-25|2025-07-28|NULL         |
|ORD100750|2025-04-24|2025-04-28|2025-05-03   |
|ORD100653|2025-09-22|2025-09-25|2025-09-26   |
|ORD100266|2025-01-29|2025-01-31|2025-02-01   |
|ORD100918|NULL      |2025-12-25|2025-01-22   |
|ORD100115|2025-09-23|2025-09-25|2025-09-28   |
|ORD100253|2025-05-30|2025-06-03|2025-06

### First Inspect the Date Columns

In [0]:
bronze_orders.select(
    F.count(
        F.when(F.col("order_date").isNull(),1)
    ).alias("null_order_date"),
    F.count(
        F.when(F.col("ship_date").isNull(),1)
    ).alias("null_ship_date"),
    F.count(
        F.when(F.col("delivery_date").isNull(),1)
    ).alias("null_delivery_date")
).show()

+---------------+--------------+------------------+
|null_order_date|null_ship_date|null_delivery_date|
+---------------+--------------+------------------+
|              7|           227|               317|
+---------------+--------------+------------------+



### Parse order_date

In [0]:
order_date_analysis=bronze_orders.withColumn(
    "parsed_order_date",
    F.try_to_date
    (
        F.col("order_date"),
         "yyyy-MM-dd"
    )
)

In [0]:
order_date_analysis.select(
    "order_id",
    "order_date",
    "parsed_order_date"
).show(truncate=False)

+---------+----------+-----------------+
|order_id |order_date|parsed_order_date|
+---------+----------+-----------------+
|ORD100218|2025-10-18|2025-10-18       |
|ORD100164|2025-12-13|2025-12-13       |
|ORD100064|2025-11-27|2025-11-27       |
|ORD100596|2025-09-06|2025-09-06       |
|ORD100313|2025-12-12|2025-12-12       |
|ORD100660|2025-01-25|2025-01-25       |
|ORD100071|2025-11-28|2025-11-28       |
|ORD100724|2025-11-08|2025-11-08       |
|ORD100851|2025-08-16|2025-08-16       |
|ORD100463|2025-07-19|2025-07-19       |
|ORD100027|2025-07-11|2025-07-11       |
|ORD100606|2025-07-25|2025-07-25       |
|ORD100750|2025-04-24|2025-04-24       |
|ORD100653|2025-09-22|2025-09-22       |
|ORD100266|2025-01-29|2025-01-29       |
|ORD100918|NULL      |NULL             |
|ORD100115|2025-09-23|2025-09-23       |
|ORD100253|2025-05-30|2025-05-30       |
|ORD100110|2025-09-13|2025-09-13       |
|ORD100031|2025-12-04|2025-12-04       |
+---------+----------+-----------------+
only showing top

### Identify Failed Date Parsing

In [0]:
invalid_order_date_df=order_date_analysis.filter(
    F.col("order_date").isNull() |
(  
    (F.trim(F.col("order_date"))!="") &
    F.col("parsed_order_date").isNull()
)
)

In [0]:
invalid_order_date_df.select(
    "order_id",
    "order_date",
    "parsed_order_date"
).show(truncate=False)

+---------+----------+-----------------+
|order_id |order_date|parsed_order_date|
+---------+----------+-----------------+
|ORD100918|NULL      |NULL             |
|ORD100862|NULL      |NULL             |
|ORD100784|NULL      |NULL             |
|ORD100902|2025/14/40|NULL             |
|ORD100856|NULL      |NULL             |
|ORD100882|2025-13-05|NULL             |
|ORD100840|2025-02-30|NULL             |
|ORD100914|NULL      |NULL             |
|ORD100839|2025-99-10|NULL             |
|ORD100912|NULL      |NULL             |
|ORD100774|NULL      |NULL             |
+---------+----------+-----------------+



### Parse All Three Date Columns

In [0]:
order_date_analysis = (
    bronze_orders.withColumn(
        "parsed_order_date",
        F.try_to_date(
            F.col("order_date"),
            'yyyy-MM-dd'
        )
    )
    .withColumn(
        "parsed_ship_date",
        F.try_to_date(
            F.col("ship_date"),
            'yyyy-MM-dd'
        )
    )
    .withColumn(
        "parsed_delivery_date",
        F.try_to_date(
            F.col("delivery_date"),
            'yyyy-MM-dd'
        )
    )
)

In [0]:
order_date_analysis.groupBy("order_status").count().show()

+------------+-----+
|order_status|count|
+------------+-----+
|     Shipped|   92|
|   Cancelled|   96|
|   Delivered|  563|
|    Returned|  104|
|      Placed|   61|
|   Confirmed|   77|
|      placed|    1|
|      shiped|    1|
|  Delivered |    2|
|    deliverd|    2|
|       Placd|    1|
+------------+-----+



### Identify Invalid ship_date

In [0]:
invalid_ship_date_df=order_date_analysis.filter(
    F.col("ship_date").isNotNull() &
    (F.trim(F.col("ship_date"))!="") &
    F.col("parsed_ship_date").isNull()
)

In [0]:
invalid_ship_date_df.select(
    "order_id",
    "ship_date",
    "parsed_ship_date"
    ).show(truncate=False)

+---------+----------+----------------+
|order_id |ship_date |parsed_ship_date|
+---------+----------+----------------+
|ORD100753|not_a_date|NULL            |
+---------+----------+----------------+



In [0]:
invalid_delivery_date_df=order_date_analysis.filter(
    F.col("delivery_date").isNull() &
    (F.trim(F.col("delivery_date")!="")) &
    F.col("parsed_delivery_date").isNull()
)

In [0]:
invalid_delivery_date_df.select(
    "order_id",
    "delivery_date",
    "parsed_delivery_date"
).show()

+--------+-------------+--------------------+
|order_id|delivery_date|parsed_delivery_date|
+--------+-------------+--------------------+
+--------+-------------+--------------------+



### Step 16 — Date Business Rules

### Rule 1 — Ship Date Cannot Be Before Order Date

In [0]:
invalid_ship_date_order_df=order_date_analysis.filter(
    F.col("parsed_order_date").isNotNull() &
    F.col("parsed_ship_date").isNotNull() &
    (F.col("parsed_ship_date") < F.col("parsed_order_date"))
)

In [0]:
invalid_ship_date_order_df.select(
    "order_id",
    "parsed_order_date",
    "parsed_ship_date"
).show()

+---------+-----------------+----------------+
| order_id|parsed_order_date|parsed_ship_date|
+---------+-----------------+----------------+
|ORD100936|       2025-09-19|      2025-09-18|
|ORD100792|       2025-12-30|      2025-12-27|
|ORD100870|       2025-10-01|      2025-09-28|
|ORD100837|       2025-03-13|      2025-03-10|
|ORD100910|       2025-05-02|      2025-04-29|
|ORD100850|       2025-02-14|      2025-02-13|
|ORD100897|       2025-08-04|      2025-08-03|
+---------+-----------------+----------------+



### Rule 2 — Delivery Date Cannot Be Before Ship Date

In [0]:
invalid_delivery_date_ship_df=order_date_analysis.filter(
    F.col("parsed_delivery_date").isNotNull() &
    F.col("parsed_ship_date").isNotNull() &
    (F.col("parsed_delivery_date")<F.col("parsed_ship_date"))
)

In [0]:
invalid_delivery_date_ship_df.select(
    "order_id",
    "parsed_ship_date",
    "parsed_delivery_date"
).show()

+---------+----------------+--------------------+
| order_id|parsed_ship_date|parsed_delivery_date|
+---------+----------------+--------------------+
|ORD100918|      2025-12-25|          2025-01-22|
|ORD100841|      2025-12-27|          2025-03-16|
|ORD100883|      2025-11-26|          2025-06-06|
|ORD100936|      2025-09-18|          2025-09-17|
|ORD100880|      2025-01-31|          2025-01-29|
|ORD100798|      2025-12-07|          2025-12-05|
|ORD100894|      2025-08-11|          2025-08-10|
|ORD100840|      2025-10-05|          2025-10-03|
|ORD100768|      2025-10-24|          2025-10-23|
|ORD100786|      2025-09-04|          2025-09-02|
|ORD100872|      2025-03-15|          2025-03-14|
+---------+----------------+--------------------+



In [0]:
order_status_reference = order_status_cleaned.select(
    "order_id",
    "order_status"
)

In [0]:
order_status_reference.show()

+---------+------------+
| order_id|order_status|
+---------+------------+
|ORD100218|     Shipped|
|ORD100164|   Cancelled|
|ORD100064|   Cancelled|
|ORD100596|   Delivered|
|ORD100313|   Delivered|
|ORD100660|   Delivered|
|ORD100071|   Delivered|
|ORD100724|    Returned|
|ORD100851|     Shipped|
|ORD100463|     Shipped|
|ORD100027|   Delivered|
|ORD100606|     Shipped|
|ORD100750|   Delivered|
|ORD100653|   Delivered|
|ORD100266|   Delivered|
|ORD100918|      Placed|
|ORD100115|   Delivered|
|ORD100253|    Returned|
|ORD100110|   Cancelled|
|ORD100031|   Delivered|
+---------+------------+
only showing top 20 rows


In [0]:
order_date_analysis = (
    order_date_analysis
    .drop("order_status")
    .join(
        order_status_reference,
        on="order_id",
        how="left"
    )
)

In [0]:
order_date_analysis.groupBy("order_status").count().show()

+------------+-----+
|order_status|count|
+------------+-----+
|     Shipped|  100|
|    Returned|  114|
|      Placed|   70|
|   Confirmed|   79|
|   Delivered|  633|
|   Cancelled|  108|
+------------+-----+



### Rule 3 — Status-Dependent Dates

In [0]:
invalid_shipped_date_df=order_date_analysis.filter(
    (F.col("order_status")=="Shipped") &
    F.col("parsed_ship_date").isNull()
)

In [0]:
invalid_shipped_date_df.select(
    "order_status",
    "parsed_ship_date"
).show()

+------------+----------------+
|order_status|parsed_ship_date|
+------------+----------------+
|     Shipped|            NULL|
|     Shipped|            NULL|
|     Shipped|            NULL|
|     Shipped|            NULL|
|     Shipped|            NULL|
|     Shipped|            NULL|
|     Shipped|            NULL|
|     Shipped|            NULL|
|     Shipped|            NULL|
|     Shipped|            NULL|
|     Shipped|            NULL|
|     Shipped|            NULL|
|     Shipped|            NULL|
+------------+----------------+



### Rule 3.2 — Delivered Order Must Have

In [0]:
invalid_deliver_date_df=order_date_analysis.filter(
    (F.col("order_status")=='Delivered') &
    F.col("parsed_delivery_date").isNull()
)

In [0]:
invalid_deliver_date_df.select(
    "order_status",
    "parsed_delivery_date"
).show()

+------------+--------------------+
|order_status|parsed_delivery_date|
+------------+--------------------+
|   Delivered|                NULL|
|   Delivered|                NULL|
|   Delivered|                NULL|
|   Delivered|                NULL|
|   Delivered|                NULL|
|   Delivered|                NULL|
|   Delivered|                NULL|
+------------+--------------------+



### Rule 3.3 — Placed Order Should Not Have Shipping or Delivery Dates

In [0]:
invalid_placed_dates_df=order_date_analysis.filter(
    (F.col("order_status") == "Placed") &
    (
    F.col("parsed_delivery_date").isNotNull()|
    F.col("parsed_ship_date").isNotNull()
    )
)

In [0]:
invalid_placed_dates_df.select(
    "order_id",
    "order_status",
    "parsed_delivery_date",
    "parsed_ship_date"
).show()

+---------+------------+--------------------+----------------+
| order_id|order_status|parsed_delivery_date|parsed_ship_date|
+---------+------------+--------------------+----------------+
|ORD100918|      Placed|          2025-01-22|      2025-12-25|
|ORD100841|      Placed|          2025-03-16|      2025-12-27|
|ORD100830|      Placed|          2025-01-24|            NULL|
|ORD100883|      Placed|          2025-06-06|      2025-11-26|
|ORD100812|      Placed|                NULL|      2025-05-19|
|ORD100844|      Placed|                NULL|      2025-06-05|
|ORD100889|      Placed|          2025-04-22|      2025-04-18|
|ORD100795|      Placed|          2025-10-30|      2025-05-09|
|ORD100941|      Placed|          2025-11-06|      2025-08-01|
|ORD100776|      Placed|          2025-10-31|      2025-10-26|
|ORD100778|      Placed|          2025-08-02|      2025-07-27|
|ORD100932|      Placed|          2025-11-23|      2025-04-03|
|ORD100228|      Placed|          2025-08-24|      2025

### Rule 3.4 — Confirmed Order

In [0]:
invalid_confirmed_delivery_df=order_date_analysis.filter(
    (F.col("order_status") == "Confirmed") &
    F.col("parsed_delivery_date").isNotNull()
)

### Rule 3.5 — Returned Order

In [0]:
invalid_returned_date_df=order_date_analysis.filter(
    (F.col("order_status") == "Returned") &
    F.col("parsed_delivery_date").isNull()
)

In [0]:
invalid_returned_date_df.show()

+---------+-----------+------------------+--------------------+--------------+----------+---------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+-----------------+----------------+--------------------+------------+
| order_id|customer_id|     customer_name|      customer_email|customer_phone|product_id|   product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|parsed_order_date|parsed_ship_date|parsed_delivery_date|order_status|
+---------+-----------+------------------+--------------------+--------------+----------+---------------+-----------+--------+----------+---------

### Step 17 — Payment Status and Order Status Business Rules

### Bring Standardized Payment Status

In [0]:
payment_status_reference = payment_status_cleaned.select(
    "order_id",
    "payment_status"
)

In [0]:
order_date_analysis = (
    order_date_analysis
    .drop("payment_status")
    .join(
        payment_status_reference,
        on="order_id",
        how="left"
    )
)

### First Understand the Combinations

In [0]:
payment_order_status_analysis=(
    order_date_analysis
    .groupBy(
        "payment_status",
        "order_status"
    )
    .count()
    .orderBy(F.desc("count"))
)
payment_order_status_analysis.show()

+--------------+------------+-----+
|payment_status|order_status|count|
+--------------+------------+-----+
|          PAID|   Delivered|  709|
|      REFUNDED|    Returned|   99|
|          PAID|     Shipped|   91|
|      REFUNDED|   Cancelled|   63|
|          PAID|      Placed|   60|
|          PAID|   Confirmed|   55|
|        FAILED|   Cancelled|   41|
|      REFUNDED|   Delivered|   40|
|          PAID|    Returned|   35|
|       PENDING|   Cancelled|   28|
|       PENDING|   Confirmed|   28|
|       PENDING|   Delivered|   24|
|       PENDING|     Shipped|   22|
|       PENDING|      Placed|   14|
|      REFUNDED|      Placed|   10|
|        FAILED|   Delivered|    4|
|      REFUNDED|     Shipped|    1|
+--------------+------------+-----+



### Business Rule — Delivered + Failed

In [0]:
invalid_delivered_failed_payment_df=order_date_analysis.filter(
    (F.col("order_status") == "Delivered") &
    (F.col("payment_status") == "FAILED")
)

In [0]:
invalid_delivered_failed_payment_df.select(
    "order_id",
    "payment_status",
    "order_status"
).show(truncate=False)

+---------+--------------+------------+
|order_id |payment_status|order_status|
+---------+--------------+------------+
|ORD100790|FAILED        |Delivered   |
|ORD100877|FAILED        |Delivered   |
|ORD100852|FAILED        |Delivered   |
|ORD100924|FAILED        |Delivered   |
+---------+--------------+------------+



### Business Rule — Delivered + Pending

In [0]:
invalid_delivered_pending_payment_df=order_date_analysis.filter(
    (F.col("order_status") == "Delivered") &
    (F.col("payment_status") == "PENDING")
)

In [0]:
invalid_delivered_pending_payment_df.select(
    "order_id",
    "payment_status",
    "order_status"
).show(truncate=False)

+---------+--------------+------------+
|order_id |payment_status|order_status|
+---------+--------------+------------+
|ORD100896|PENDING       |Delivered   |
|ORD100846|PENDING       |Delivered   |
|ORD100843|PENDING       |Delivered   |
|ORD100853|PENDING       |Delivered   |
|ORD100890|PENDING       |Delivered   |
|ORD100843|PENDING       |Delivered   |
|ORD100853|PENDING       |Delivered   |
|ORD100900|PENDING       |Delivered   |
|ORD100820|PENDING       |Delivered   |
|ORD100908|PENDING       |Delivered   |
|ORD100781|PENDING       |Delivered   |
|ORD100939|PENDING       |Delivered   |
|ORD100843|PENDING       |Delivered   |
|ORD100853|PENDING       |Delivered   |
|ORD100843|PENDING       |Delivered   |
|ORD100853|PENDING       |Delivered   |
|ORD100843|PENDING       |Delivered   |
|ORD100853|PENDING       |Delivered   |
|ORD100843|PENDING       |Delivered   |
|ORD100853|PENDING       |Delivered   |
+---------+--------------+------------+
only showing top 20 rows


### Business Rule — Placed + Refunded

In [0]:
invalid_placed_refunded_df=order_date_analysis.filter(
    (F.col("order_status")=="Placed") &
    (F.col("payment_status")=="REFUNDED")
)

In [0]:
invalid_placed_refunded_df.select(
    "order_id",
    "payment_status",
    "order_status"
).show(truncate=False)

+---------+--------------+------------+
|order_id |payment_status|order_status|
+---------+--------------+------------+
|ORD100812|REFUNDED      |Placed      |
|ORD100844|REFUNDED      |Placed      |
|ORD100889|REFUNDED      |Placed      |
|ORD100776|REFUNDED      |Placed      |
|ORD100778|REFUNDED      |Placed      |
|ORD100832|REFUNDED      |Placed      |
|ORD100884|REFUNDED      |Placed      |
|ORD100921|REFUNDED      |Placed      |
|ORD100786|REFUNDED      |Placed      |
|ORD100791|REFUNDED      |Placed      |
+---------+--------------+------------+



### Create the Business-Rule Flag

In [0]:
order_date_analysis=order_date_analysis.withColumn(
    "payment_order_status_violation",
    F.when(
    (
        (F.col("order_status")== "Delivered") &
        (F.col("payment_status")=="FAILED")
    ) |
    (
        (F.col("order_status")== "Delivered") &
        (F.col("payment_status")=="PENDING")
    ) |
    (
        (F.col("order_status") == "Placed") &
        (F.col("payment_status") == "REFUNDED")
    ),
    F.lit(True)
    ).otherwise(F.lit(False))
)

In [0]:
order_date_analysis.filter(
    F.col("payment_order_status_violation")==True
).select(
    "order_id",
    "payment_status",
    "order_status"
).show()

+---------+--------------+------------+
| order_id|payment_status|order_status|
+---------+--------------+------------+
|ORD100896|       PENDING|   Delivered|
|ORD100846|       PENDING|   Delivered|
|ORD100812|      REFUNDED|      Placed|
|ORD100844|      REFUNDED|      Placed|
|ORD100889|      REFUNDED|      Placed|
|ORD100843|       PENDING|   Delivered|
|ORD100853|       PENDING|   Delivered|
|ORD100790|        FAILED|   Delivered|
|ORD100890|       PENDING|   Delivered|
|ORD100877|        FAILED|   Delivered|
|ORD100776|      REFUNDED|      Placed|
|ORD100852|        FAILED|   Delivered|
|ORD100778|      REFUNDED|      Placed|
|ORD100843|       PENDING|   Delivered|
|ORD100853|       PENDING|   Delivered|
|ORD100900|       PENDING|   Delivered|
|ORD100820|       PENDING|   Delivered|
|ORD100832|      REFUNDED|      Placed|
|ORD100884|      REFUNDED|      Placed|
|ORD100908|       PENDING|   Delivered|
+---------+--------------+------------+
only showing top 20 rows


### Step 18 — Shipping City Validation

### Investigate Missing or Blank Cities

In [0]:
invalid_shipping_city_df=bronze_orders.filter(
    F.col("shipping_city").isNull() |
    (
        F.col("shipping_city").isNotNull() &
        (F.trim(F.col("shipping_city"))=="")
    )
)

In [0]:
invalid_shipping_city_df.select(
    "order_id",
    "shipping_city",
    "shipping_state",
    "pincode"
).show(truncate=False)

+---------+-------------+--------------+-------+
|order_id |shipping_city|shipping_state|pincode|
+---------+-------------+--------------+-------+
|ORD100929|NULL         |NULL          |226007 |
|ORD100937|NULL         |NULL          |500075 |
|ORD100849|             |Tamil Nadu    |641054 |
|ORD100947|NULL         |NULL          |682035 |
|ORD100792|             |Telangana     |500050 |
|ORD100870|NULL         |NULL          |560060 |
|ORD100758|             |Rajasthan     |302003 |
|ORD100886|NULL         |NULL          |500096 |
|ORD100858|NULL         |NULL          |682015 |
|ORD100847|             |Tamil Nadu    |625099 |
|ORD100886|NULL         |NULL          |500096 |
|ORD100945|NULL         |NULL          |110043 |
|ORD100944|             |West Bengal   |700003 |
|ORD100882|NULL         |NULL          |700050 |
|ORD100878|NULL         |NULL          |700021 |
|ORD100768|             |Telangana     |500048 |
|ORD100950|             |Tamil Nadu    |625065 |
|ORD100811|NULL     

In [0]:
shipping_city_analysis=bronze_orders.withColumn(
    "shipping_city",
    F.initcap(
        F.trim(F.col("shipping_city"))
    )
)

In [0]:
shipping_city_analysis = (
    bronze_orders.withColumn(
        "shipping_city",
        F.when(
            F.trim(F.col("shipping_city"))=="",
            None
        ).otherwise(F.initcap(F.trim(F.col("shipping_city"))))
    )
)

### Recheck the Result

In [0]:
shipping_city_analysis.select(
    "order_id",
    "shipping_city"
).show(truncate=False)

+---------+-------------+
|order_id |shipping_city|
+---------+-------------+
|ORD100218|Bengaluru    |
|ORD100164|Lucknow      |
|ORD100064|Hyderabad    |
|ORD100596|Pune         |
|ORD100313|Madurai      |
|ORD100660|Jaipur       |
|ORD100071|Kolkata      |
|ORD100724|Kolkata      |
|ORD100851|Kolkata      |
|ORD100463|Lucknow      |
|ORD100027|Ahmedabad    |
|ORD100606|Bengaluru    |
|ORD100750|Coimbatore   |
|ORD100653|Lucknow      |
|ORD100266|Madurai      |
|ORD100918|Delhi        |
|ORD100115|Delhi        |
|ORD100253|Kolkata      |
|ORD100110|Bengaluru    |
|ORD100031|Lucknow      |
+---------+-------------+
only showing top 20 rows


In [0]:
shipping_city_analysis.filter(
    F.col("pincode")=="625099"
).select(
    "shipping_city",
    "shipping_state"
).show()

+-------------+--------------+
|shipping_city|shipping_state|
+-------------+--------------+
|         NULL|    Tamil Nadu|
|      Madurai|    Tamil Nadu|
|      Madurai|    Tamil Nadu|
+-------------+--------------+



In [0]:
invalid_shipping_city_df=shipping_city_analysis.filter(
    F.col("shipping_city").isNull()
)
invalid_shipping_city_df.select(
    "order_id",
    "shipping_city"
).show(truncate=False)

+---------+-------------+
|order_id |shipping_city|
+---------+-------------+
|ORD100929|NULL         |
|ORD100937|NULL         |
|ORD100849|NULL         |
|ORD100947|NULL         |
|ORD100792|NULL         |
|ORD100870|NULL         |
|ORD100758|NULL         |
|ORD100886|NULL         |
|ORD100858|NULL         |
|ORD100847|NULL         |
|ORD100886|NULL         |
|ORD100945|NULL         |
|ORD100944|NULL         |
|ORD100882|NULL         |
|ORD100878|NULL         |
|ORD100768|NULL         |
|ORD100950|NULL         |
|ORD100811|NULL         |
|ORD100751|NULL         |
|ORD100864|NULL         |
+---------+-------------+
only showing top 20 rows


### Check pincode-to-city consistency

In [0]:
pincode_city_analysis=(
    bronze_orders.filter(
        F.col("pincode").isNotNull() &
        (F.trim(F.col("pincode"))!="") &
        F.col("shipping_city").isNotNull() &
        (F.trim(F.col("shipping_city"))!="")
    ).select(
        "pincode",
        F.trim(F.col("shipping_city")).alias("shipping_city")
    )
    .groupBy("pincode")
    .agg(
    F.countDistinct("shipping_city").alias("city_count")
    )
    .orderBy(F.desc("city_count"))
)
pincode_city_analysis.show()

+-------+----------+
|pincode|city_count|
+-------+----------+
|  60001|         2|
| 302050|         2|
| 226001|         1|
| 560050|         1|
| 380023|         1|
| 641080|         1|
| 110009|         1|
| 700036|         1|
| 700060|         1|
| 700008|         1|
| 226094|         1|
| 302000|         1|
| 625013|         1|
| 560014|         1|
| 560073|         1|
| 226096|         1|
| 411008|         1|
| 625051|         1|
| 500032|         1|
| 110020|         1|
+-------+----------+
only showing top 20 rows


### Step 19 — Shipping State Validation and Standardization

### Investigate Distinct State Values

In [0]:
shipping_state_analysis = (
    bronze_orders.select(
        F.trim(F.col("shipping_state")).alias("shipping_state")
    )
    .groupBy("shipping_state")
    .count()
    .orderBy(F.desc("count"))
)
shipping_state_analysis.show(truncate=False)

+--------------+-----+
|shipping_state|count|
+--------------+-----+
|Tamil Nadu    |227  |
|Maharashtra   |133  |
|Telangana     |91   |
|Gujarat       |87   |
|Rajasthan     |86   |
|Uttar Pradesh |80   |
|Karnataka     |78   |
|Kerala        |75   |
|Delhi         |67   |
|West Bengal   |61   |
|NULL          |15   |
+--------------+-----+



### Identify Missing States

In [0]:
invalid_shipping_state_df=bronze_orders.filter(
    F.col("shipping_state").isNull() |
    (F.trim(F.col("shipping_state"))=="")
)
invalid_shipping_state_df.select(
    "order_id",
    "shipping_city",
    "shipping_state",
    "pincode"
).show(truncate=False)

+---------+-------------+--------------+-------+
|order_id |shipping_city|shipping_state|pincode|
+---------+-------------+--------------+-------+
|ORD100929|NULL         |NULL          |226007 |
|ORD100937|NULL         |NULL          |500075 |
|ORD100947|NULL         |NULL          |682035 |
|ORD100870|NULL         |NULL          |560060 |
|ORD100886|NULL         |NULL          |500096 |
|ORD100858|NULL         |NULL          |682015 |
|ORD100886|NULL         |NULL          |500096 |
|ORD100945|NULL         |NULL          |110043 |
|ORD100882|NULL         |NULL          |700050 |
|ORD100878|NULL         |NULL          |700021 |
|ORD100811|NULL         |NULL          |302096 |
|ORD100751|NULL         |NULL          |641086 |
|ORD100864|NULL         |NULL          |641054 |
|ORD100791|NULL         |NULL          |110054 |
|ORD100899|NULL         |NULL          |411083 |
+---------+-------------+--------------+-------+



### Standardize State Formatting

In [0]:
shipping_state_analysis= bronze_orders.withColumn(
    "shipping_state",
    F.when(
        F.trim(F.col("shipping_state"))=="",
        None
    ).otherwise(
        F.initcap(
            F.trim(F.col("shipping_state"))
        )
    )
)

In [0]:
shipping_state_analysis.groupBy("shipping_state").count().show()

+--------------+-----+
|shipping_state|count|
+--------------+-----+
|     Karnataka|   78|
| Uttar Pradesh|   80|
|     Telangana|   91|
|   Maharashtra|  133|
|    Tamil Nadu|  227|
|     Rajasthan|   86|
|   West Bengal|   61|
|       Gujarat|   87|
|         Delhi|   67|
|        Kerala|   75|
|          NULL|   15|
+--------------+-----+



### Step 20 — Pincode Validation

### Identify Missing Pincodes

In [0]:
invalid_pincode_df=bronze_orders.filter(
    F.col("pincode").isNull() |
    (F.trim(F.col("pincode"))=="")
)
invalid_pincode_df.select(
    "order_id",
    "shipping_city",
    "shipping_state",
    "pincode"
).show(truncate=False)

+---------+-------------+--------------+-------+
|order_id |shipping_city|shipping_state|pincode|
+---------+-------------+--------------+-------+
|ORD100773|Kochi        |Kerala        |NULL   |
|ORD100943|Chennai      |Tamil Nadu    |NULL   |
|ORD100871|Pune         |Maharashtra   |NULL   |
|ORD100881|Bengaluru    |Karnataka     |NULL   |
|ORD100920|Madurai      |Tamil Nadu    |NULL   |
|ORD100892|Pune         |Maharashtra   |NULL   |
|ORD100769|Mumbai       |Maharashtra   |NULL   |
|ORD100916|Madurai      |Tamil Nadu    |NULL   |
|ORD100848|Pune         |Maharashtra   |NULL   |
|ORD100821|Coimbatore   |Tamil Nadu    |NULL   |
|ORD100793|Jaipur       |Rajasthan     |NULL   |
+---------+-------------+--------------+-------+



### Identify Invalid Pincode Formats

In [0]:
pincode_format=r"^[0-9]{6}$"
invalid_pincode_format_df=bronze_orders.filter(
    F.col("pincode").isNotNull() &
    (F.trim(F.col("pincode"))!="") &
    ~F.trim(F.col("pincode")).rlike(pincode_format)
)

invalid_pincode_format_df.select(
    "order_id",
    "shipping_city",
    "shipping_state",
    "pincode"
).show(truncate=False)

+---------+-------------+--------------+-------+
|order_id |shipping_city|shipping_state|pincode|
+---------+-------------+--------------+-------+
|ORD100851|Kolkata      |West Bengal   |70012  |
|ORD100862|Delhi        |Delhi         |1234567|
|ORD100826|Madurai      |Tamil Nadu    |ABC123 |
|ORD100851|Kolkata      |West Bengal   |70012  |
|ORD100780|Delhi        |Delhi         |60001  |
|ORD100909|Jaipur       |Rajasthan     |60001  |
|ORD100875|Jaipur       |Rajasthan     |60001  |
+---------+-------------+--------------+-------+



### Step 21 — Exact Duplicate Analysis

In [0]:
exact_duplicate_analysis = (
    bronze_orders
    .groupBy(bronze_orders.columns)
    .count()
    .filter(F.col("count")>1)
    .orderBy(F.desc("count"))
)
exact_duplicate_analysis.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+-----+
| order_id|customer_id|    customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|count|
+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+-------

### Find the Number of Exact Duplicate Records

In [0]:
exact_duplicate_count = (
    exact_duplicate_analysis
    .agg(
        F.sum(F.col("count")-1).alias("duplicate_rows")
    )
)
exact_duplicate_count.show()

+--------------+
|duplicate_rows|
+--------------+
|            26|
+--------------+



### Inspect the Exact Duplicates

In [0]:
exact_duplicate_orders = (
    bronze_orders
    .groupBy(bronze_orders.columns)
    .count()
    .filter(F.col("count")>1)
)

exact_duplicate_orders.select(
    "order_id",
    "customer_id",
    "customer_name",
    "product_id",
    "quantity",
    "unit_price",
    "order_amount",
    "final_amount",
    "order_status",
    "count"
).show(exact_duplicate_orders.count(),truncate=False)

+---------+-----------+-----------------+----------+--------+----------+------------+------------+------------+-----+
|order_id |customer_id|customer_name    |product_id|quantity|unit_price|order_amount|final_amount|order_status|count|
+---------+-----------+-----------------+----------+--------+----------+------------+------------+------------+-----+
|ORD100851|CUST10128  |Rahul Subramaniam|PROD1096  |10      |4238.02   |42380.2     |42380.2     |Shipped     |2    |
|ORD100027|CUST10008  |Lakshmi Balan    |PROD1006  |6       |4858.4    |29150.4     |26235.36    |Delivered   |2    |
|ORD100750|CUST10139  |Sunil Shetty     |PROD1007  |4       |2854.72   |11418.88    |5709.44     |Delivered   |2    |
|ORD100534|CUST10024  |Ayesha Chandran  |PROD1003  |8       |23912.69  |191301.52   |191301.52   |Cancelled   |2    |
|ORD100465|CUST10135  |Manisha Nair     |PROD1039  |3       |468.35    |1405.05     |562.02      |Returned    |2    |
|ORD100289|CUST10030  |Ajay Verma       |PROD1054  |8   

### Step 22 — Same order_id With Different Records

In [0]:
order_id_duplicate_analysis = (
    bronze_orders
    .groupBy("order_id")
    .count()
    .filter(F.col("count")>1)
    .orderBy(F.desc("count"))
)
order_id_duplicate_analysis.show(truncate=False)

+---------+-----+
|order_id |count|
+---------+-----+
|ORD100750|3    |
|ORD100456|3    |
|ORD100363|2    |
|ORD100209|2    |
|ORD100534|2    |
|ORD100041|2    |
|ORD100200|2    |
|ORD100819|2    |
|ORD100007|2    |
|ORD100763|2    |
|ORD100851|2    |
|ORD100113|2    |
|ORD100443|2    |
|ORD100130|2    |
|ORD100843|2    |
|ORD100465|2    |
|ORD100027|2    |
|ORD100585|2    |
|ORD100724|2    |
|ORD100289|2    |
+---------+-----+
only showing top 20 rows


### Inspect the Duplicate Ordersb

In [0]:
duplicate_orders_ids=order_id_duplicate_analysis.select("order_id")

duplicate_order_records = (
    bronze_orders
    .join(
        duplicate_orders_ids,
        on="order_id",
        how="inner"
    )
    .orderBy(F.desc("order_id"))
)
# print(duplicate_order_records.count())
duplicate_order_records.show(duplicate_order_records.count(),truncate=False)

+---------+-----------+-----------------+------------------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------------+---------------+--------+-------------+-----------+
|order_id |customer_id|customer_name    |customer_email                |customer_phone|product_id|product_name       |category   |quantity|unit_price|discount_percent|order_amount|final_amount|payment_method  |payment_status|order_status|order_date|ship_date |delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp       |source_file    |batch_id|source_system|pipeline_id|
+---------+-----------+-----------------+------------------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+----------

### Check Whether the Difference Is Real

In [0]:
order_id_variation_analysis = (
    bronze_orders
    .groupBy("order_id")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("customer_id").alias("customer_id_count"),
        F.countDistinct("product_id").alias("product_id_count"),
        F.countDistinct("quantity").alias("quantity_count"),
        F.countDistinct("unit_price").alias("unit_price_count"),
        F.countDistinct("discount_percent").alias("discount_count"),
        F.countDistinct("order_amount").alias("order_amount_count"),
        F.countDistinct("final_amount").alias("final_amount_count"),
        F.countDistinct("order_status").alias("order_status_count"),
    )
)

order_id_variation_analysis.show(truncate=False)

+---------+------------+-----------------+----------------+--------------+----------------+--------------+------------------+------------------+------------------+
|order_id |record_count|customer_id_count|product_id_count|quantity_count|unit_price_count|discount_count|order_amount_count|final_amount_count|order_status_count|
+---------+------------+-----------------+----------------+--------------+----------------+--------------+------------------+------------------+------------------+
|ORD100218|1           |1                |1               |1             |1               |1             |1                 |1                 |1                 |
|ORD100164|1           |1                |1               |1             |1               |1             |1                 |1                 |1                 |
|ORD100064|1           |1                |1               |1             |1               |1             |1                 |1                 |1                 |
|ORD100596|1    

### Step 23 — Resolve Conflicting order_id Records

### Create a Completeness Score

In [0]:
duplicate_resolution_analysis = (
    bronze_orders
    .withColumn(
        "completeness_score",
        sum(
            F.when(
                F.col(column).isNotNull() &
                (F.trim(F.col(column).cast("string"))!=""),
                1
            ).otherwise(0)
            for column in bronze_orders.columns
        )
    )
)
duplicate_resolution_analysis.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+------------------+
| order_id|customer_id|    customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|completeness_score|
+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------

### Rank Records Within Each order_id

In [0]:
from pyspark.sql.window import Window

order_window = Window.partitionBy(
    "order_id"
).orderBy(F.desc("completeness_score"))

duplicate_resolution_analysis = (
    duplicate_resolution_analysis
    .withColumn(
        "record_rank",
        F.row_number().over(order_window)
    )
)
duplicate_resolution_analysis.show(truncate=False)

+---------+-----------+-----------------+------------------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------------+---------------+--------+-------------+-----------+------------------+-----------+
|order_id |customer_id|customer_name    |customer_email                |customer_phone|product_id|product_name       |category   |quantity|unit_price|discount_percent|order_amount|final_amount|payment_method  |payment_status|order_status|order_date|ship_date |delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp       |source_file    |batch_id|source_system|pipeline_id|completeness_score|record_rank|
+---------+-----------+-----------------+------------------------------+--------------+----------+-------------------+-----------+--------+----------+--------

### Step 24 — Identify Multi-Error Records

### Create a Validation Flag for Each Rule

In [0]:
orders_quality=bronze_orders.withColumn(
    "quantity_invalid",
    F.when(
        F.col("quantity").isNull() |
        (F.col("quantity") <=0),
        True
    ).otherwise(False)
)

In [0]:
orders_quality = orders_quality.withColumn(
    "unit_price_invalid",
    F.when(
        F.col("unit_price").isNull() |
        (F.col("unit_price")<=0),
        True
    ).otherwise(False)
)

### Step 24 — Create Validation Flags.

In [0]:
validation_df=bronze_orders

In [0]:
validation_df.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|    customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+--

In [0]:
# from pyspark.sql import Window
# from pyspark.sql import functions as F

# def dedupe_by_order_id(df, tiebreak_col="order_id"):
#     """
#     Keeps exactly one row per order_id.
#     Replace the orderBy() below with your real survivorship rule.
#     """
#     window_spec = Window.partitionBy("order_id").orderBy(
#         F.monotonically_increasing_id().desc()   # <-- "keep the last-seen row"; change this
#     )
#     return (
#         df
#         .withColumn("_rn", F.row_number().over(window_spec))
#         .filter(F.col("_rn") == 1)
#         .drop("_rn")
#     )

In [0]:
# payment_status_cleaned_deduped   = dedupe_by_order_id(payment_status_cleaned)
# order_status_cleaned_deduped     = dedupe_by_order_id(order_status_cleaned)
# order_with_payment_method_deduped = dedupe_by_order_id(order_with_payment_method)
# orders_with_category_deduped     = dedupe_by_order_id(orders_with_category)

In [0]:
validation_df = (
    validation_df
    .drop("payment_status","order_status","payment_method","category")
    .join(
        payment_status_cleaned
        .select("order_id", "payment_status")
        .dropDuplicates(['order_id']),
        on="order_id", how="left"
    )
    .join(
        order_status_cleaned
        .select("order_id", "order_status")
        .dropDuplicates(['order_id']),
        on="order_id", how="left"
    )
    .join(
        order_with_payment_method
        .select("order_id", "payment_method")
        .dropDuplicates(['order_id']),
        on="order_id", how="left"
    )
    .join(
        orders_with_category
        .select("order_id", "category")
        .dropDuplicates(['order_id']),
        on="order_id", how="left"
    )
)

In [0]:
# validation_df = (
#     validation_df
#     .drop("order_status")
#     .join(
#         order_status_cleaned_deduped.select("order_id", "order_status"),
#         on="order_id", how="left"
#     )
# )

In [0]:
# validation_df = (
#     validation_df
#     .drop("payment_method")
#     .join(
#         order_with_payment_method_deduped.select("order_id", "payment_method"),
#         on="order_id", how="left"
#     )
# )

In [0]:
# validation_df = (
#     validation_df
#     .drop("category")
#     .join(
#         orders_with_category_deduped.select("order_id", "category"),
#         on="order_id", how="left"
#     )
# )

In [0]:
validation_df.count()

1000

In [0]:
validation_df.select("category","payment_method","payment_status","order_status",).show()

+-----------+----------------+--------------+------------+
|   category|  payment_method|payment_status|order_status|
+-----------+----------------+--------------+------------+
|     SPORTS|     NET BANKING|          PAID|     Shipped|
|ELECTRONICS|     NET BANKING|      REFUNDED|   Cancelled|
|      BOOKS|      DEBIT CARD|      REFUNDED|   Cancelled|
|       HOME|             UPI|          PAID|   Delivered|
|    FASHION|     NET BANKING|          PAID|   Delivered|
|ELECTRONICS|             UPI|          PAID|   Delivered|
|       HOME|          WALLET|          PAID|   Delivered|
|     SPORTS|CASH ON DELIVERY|          PAID|    Returned|
|ELECTRONICS|          WALLET|          PAID|     Shipped|
|ELECTRONICS|CASH ON DELIVERY|       PENDING|     Shipped|
|ELECTRONICS|      DEBIT CARD|          PAID|   Delivered|
|     BEAUTY|             UPI|          PAID|     Shipped|
|ELECTRONICS|          WALLET|          PAID|   Delivered|
|    GROCERY|     NET BANKING|          PAID|   Delivere

### Step 24.2 — Customer Name, Email and Phone Validation Flags

### Invalid Customer Name

In [0]:
validation_df = (
    validation_df
    .join(
        unrecoverable_customer_df
        .select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_customer_name",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_customer_name":False
    })
)

### Invalid Customer Email

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_email_df
        .select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_customer_email",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_customer_email":False
    })
)

### Invalid Customer Phone

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_phone_df
        .select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_customer_phone",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_customer_phone":False
    })
)

In [0]:
validation_df.select(
    "order_id",
    "invalid_customer_name",
    "invalid_customer_email",
    "invalid_customer_phone"
).show(truncate=False)

+---------+---------------------+----------------------+----------------------+
|order_id |invalid_customer_name|invalid_customer_email|invalid_customer_phone|
+---------+---------------------+----------------------+----------------------+
|ORD100218|false                |false                 |false                 |
|ORD100164|false                |false                 |false                 |
|ORD100064|false                |false                 |false                 |
|ORD100596|false                |false                 |false                 |
|ORD100313|false                |false                 |false                 |
|ORD100660|false                |false                 |false                 |
|ORD100071|false                |false                 |false                 |
|ORD100724|false                |false                 |false                 |
|ORD100851|false                |false                 |false                 |
|ORD100463|false                |false  

### 24.3 — Order ID, Customer ID and Product ID validation flags

### Invalid Order ID

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_order_id_df
        .select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_order_id",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_order_id":False
    })
)

### Invalid Customer ID

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_customer_id_df
        .select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_customer_id",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_customer_id":False
    })
)

### Invalid Product ID

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_product_id_df.select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_product_id",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_product_id":False
    })
)

In [0]:
validation_df.select(
    "order_id",
    "customer_id",
    "product_id",
    "invalid_order_id",
    "invalid_customer_id",
    "invalid_product_id"
).show()

+---------+-----------+----------+----------------+-------------------+------------------+
| order_id|customer_id|product_id|invalid_order_id|invalid_customer_id|invalid_product_id|
+---------+-----------+----------+----------------+-------------------+------------------+
|ORD100218|  CUST10020|  PROD1064|           false|              false|             false|
|ORD100164|  CUST10042|  PROD1073|           false|              false|             false|
|ORD100064|  CUST10136|  PROD1066|           false|              false|             false|
|ORD100596|  CUST10067|  PROD1036|           false|              false|             false|
|ORD100313|  CUST10123|  PROD1075|           false|              false|             false|
|ORD100660|  CUST10016|  PROD1048|           false|              false|             false|
|ORD100071|  CUST10105|  PROD1100|           false|              false|             false|
|ORD100724|  CUST10036|  PROD1020|           false|              false|             false|

### Step 24.4 — Product name and category validation flags.

### Invalid Product Name

In [0]:
invalid_product_name_df=orders_with_product_name.filter(
    F.col("product_name").isNull() |
    (F.trim(F.col("product_name"))=="")
).select("order_id").dropDuplicates(["order_id"])

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_product_name_df
        .withColumn(
            "invalid_product_name",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_product_name":False
    })
)

In [0]:
validation_df.select(
    "order_id",
    "product_id",
    "product_name",
    "invalid_product_name"
).show()

+---------+----------+-------------------+--------------------+
| order_id|product_id|       product_name|invalid_product_name|
+---------+----------+-------------------+--------------------+
|ORD100218|  PROD1064|        Cricket Bat|               false|
|ORD100164|  PROD1073|              Mouse|               false|
|ORD100064|  PROD1066|              Books|               false|
|ORD100596|  PROD1036|       Water Bottle|               false|
|ORD100313|  PROD1075|       Cotton Shirt|               false|
|ORD100660|  PROD1048|  Bluetooth Speaker|               false|
|ORD100071|  PROD1100|      Mixer Grinder|               false|
|ORD100724|  PROD1020|        Cricket Bat|               false|
|ORD100851|  PROD1096|    Laptop Backpack|               false|
|ORD100463|  PROD1049|        LED Monitor|               false|
|ORD100027|  PROD1006|           Keyboard|               false|
|ORD100606|  PROD1081|   Herbal Face Wash|               false|
|ORD100750|  PROD1007|              Mous

In [0]:
validation_df = (
    validation_df
    .join(
    invalid_category_df
    .select("order_id")
    .dropDuplicates(["order_id"])
    .withColumn("invalid_category",F.lit(True)),
    on="order_id",
    how="left"
    )
    .fillna({
        "invalid_category":False
    })
)

In [0]:
validation_df.select(
    "order_id",
    "category",
    "invalid_category"
).show()

+---------+-----------+----------------+
| order_id|   category|invalid_category|
+---------+-----------+----------------+
|ORD100218|     SPORTS|           false|
|ORD100164|ELECTRONICS|           false|
|ORD100064|      BOOKS|           false|
|ORD100596|       HOME|           false|
|ORD100313|    FASHION|           false|
|ORD100660|ELECTRONICS|           false|
|ORD100071|       HOME|           false|
|ORD100724|     SPORTS|           false|
|ORD100851|ELECTRONICS|           false|
|ORD100463|ELECTRONICS|           false|
|ORD100027|ELECTRONICS|           false|
|ORD100606|     BEAUTY|           false|
|ORD100750|ELECTRONICS|           false|
|ORD100653|    GROCERY|           false|
|ORD100266|    FASHION|           false|
|ORD100918|    FASHION|           false|
|ORD100115|ELECTRONICS|           false|
|ORD100253|ELECTRONICS|           false|
|ORD100110|    FASHION|           false|
|ORD100031|ELECTRONICS|           false|
+---------+-----------+----------------+
only showing top

### Step 24.5 — Quantity, Unit Price, and Discount Validation Flags

### Invalid Quantity Flag

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_quantity_df
        .select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_quantity",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_quantity":False
    })
)

### Invalid Unit Price Flag

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_unit_price_df
        .select("order_id")
        .dropDuplicates()
        .withColumn("invalid_unit_price",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_unit_price":False
    })
)

### Invalid Discount Flag

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_discount_df.select("order_id")
        .dropDuplicates()
        .withColumn("invalid_discount_percent",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_discount_percent":False
    })
)

### Check the Three Flags

In [0]:
validation_df.select(
    "order_id",
    "quantity",
    "unit_price",
    "discount_percent",
    "invalid_quantity",
    "invalid_unit_price",
    "invalid_discount_percent"
).show()

+---------+--------+----------+----------------+----------------+------------------+------------------------+
| order_id|quantity|unit_price|discount_percent|invalid_quantity|invalid_unit_price|invalid_discount_percent|
+---------+--------+----------+----------------+----------------+------------------+------------------------+
|ORD100218|       8|   5037.33|               0|           false|             false|                   false|
|ORD100164|       4|   2091.76|              80|           false|             false|                   false|
|ORD100064|       4|   2731.88|              10|           false|             false|                   false|
|ORD100596|       9|    334.78|              80|           false|             false|                   false|
|ORD100313|      10|   2379.61|              50|           false|             false|                   false|
|ORD100660|       5|   7784.98|              10|           false|             false|                   false|
|ORD100071

### Step 24.6 — Order Amount and Final Amount Validation Flags

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_order_amount_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_order_amount",F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_order_amount":False
    })
)

### Invalid Final Amount Flag

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_final_amount_df
        .select("order_id")
        .dropDuplicates()
        .withColumn("invalid_final_amount",F.lit(True)),
        on="order_id",
        how="left"
)
    .fillna({
        "invalid_final_amount":False
    })
)

In [0]:
validation_df.select(
    "order_amount",
    "order_amount",
    "final_amount",
    "invalid_order_amount",
    "invalid_final_amount"
).show()

+------------+------------+------------+--------------------+--------------------+
|order_amount|order_amount|final_amount|invalid_order_amount|invalid_final_amount|
+------------+------------+------------+--------------------+--------------------+
|    40298.64|    40298.64|    40298.64|               false|               false|
|     8367.04|     8367.04|     1673.41|               false|               false|
|    10927.52|    10927.52|     9834.77|               false|               false|
|     3013.02|     3013.02|       602.6|               false|               false|
|     23796.1|     23796.1|    11898.05|               false|               false|
|     38924.9|     38924.9|    35032.41|               false|               false|
|    52348.56|    52348.56|    52348.56|               false|               false|
|     33014.1|     33014.1|    13205.64|               false|               false|
|     42380.2|     42380.2|     42380.2|               false|               false|
|   

In [0]:
validation_df.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+--------+----------+----------------+------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+--------------+------------+----------------+-----------+---------------------+----------------------+----------------------+----------------+-------------------+------------------+--------------------+----------------+----------------+------------------+------------------------+--------------------+--------------------+
| order_id|customer_id|    customer_name|      customer_email|customer_phone|product_id|       product_name|quantity|unit_price|discount_percent|order_amount|final_amount|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|payment_status|order_status|  payment_method|

In [0]:
order_with_payment_method.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+-------------------------+
| order_id|customer_id|    customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|normalized_payment_method|
+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+

In [0]:
invalid_payment_method_df=(
    order_with_payment_method
    .filter(
        F.col("payment_method").isNull() |
        (F.trim(F.col('payment_method'))=="")
    )
    .select("order_id")
    .dropDuplicates(["order_id"])
)
invalid_payment_method_df.show()

+--------+
|order_id|
+--------+
+--------+



In [0]:
validation_df = (
    validation_df
    .join(
        invalid_payment_method_df
        .withColumn("invalid_payment_method",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_payment_method":False
    })
)

In [0]:
invalid_payment_status_df = (
    payment_status_cleaned.filter(
        ~F.col("payment_status").isin(
            "PAID",
            "REFUNDED",
            "PENDING",
            "FAILED"
        )
    ).select("order_id").dropDuplicates()
)
invalid_payment_status_df.show()

+--------+
|order_id|
+--------+
+--------+



In [0]:
validation_df = (
    validation_df
    .join(
        invalid_payment_status_df
        .withColumn(
            "invalid_payment_status",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_payment_status": False
    })
)

### Invalid Order Status

In [0]:
invalid_order_status_df_final = (
    order_status_cleaned.filter(
        ~F.col("order_status").isin(
            "Shipped",
            "Cancelled",
            "Delivered",
            "Returned",
            "Placed",
            "Confirmed"
        )
    )
    .select("order_id")
    .dropDuplicates()
)
invalid_order_status_df_final.show()

+--------+
|order_id|
+--------+
+--------+



In [0]:
validation_df = (
    validation_df
    .join(
        invalid_order_status_df_final
        .withColumn(
            "invalid_order_status",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_order_status": False
    })
)

In [0]:
validation_df.select(
    "order_id",
    "payment_method",
    "payment_status",
    "order_status",
    "invalid_payment_method",
    "invalid_payment_status",
    "invalid_order_status"
).show(truncate=False)

+---------+----------------+--------------+------------+----------------------+----------------------+--------------------+
|order_id |payment_method  |payment_status|order_status|invalid_payment_method|invalid_payment_status|invalid_order_status|
+---------+----------------+--------------+------------+----------------------+----------------------+--------------------+
|ORD100218|NET BANKING     |PAID          |Shipped     |false                 |false                 |false               |
|ORD100164|NET BANKING     |REFUNDED      |Cancelled   |false                 |false                 |false               |
|ORD100064|DEBIT CARD      |REFUNDED      |Cancelled   |false                 |false                 |false               |
|ORD100596|UPI             |PAID          |Delivered   |false                 |false                 |false               |
|ORD100313|NET BANKING     |PAID          |Delivered   |false                 |false                 |false               |
|ORD1006

### Step 24.8 — Date Validation Flags

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_order_date_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_order_date",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_order_date":False
    })
)

### Invalid Ship Date

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_ship_date_df
        .select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_ship_date",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_ship_date":False
    })
)

### Invalid Delivery Date

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_deliver_date_df.select("order_id").dropDuplicates()
        .withColumn("invalid_delivery_date",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_delivery_date":False
    })
)

### Ship Date Before Order Date

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_ship_date_order_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_ship_date_order",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_ship_date_order": False
    })
)

### Delivery Date Before Ship Date

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_delivery_date_ship_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_delivery_date_ship",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_delivery_date_ship": False
    })
)

### 24.8.6 — Status-Dependent Date Rules

### Shipped without ship date

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_shipped_date_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_shipped_date",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_shipped_date": False
    })
)

### Delivered without delivery date

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_deliver_date_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_deliver_date",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_deliver_date": False
    })
)

### Placed with shipping/delivery dates

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_placed_dates_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_placed_dates",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_placed_dates": False
    })
)

### Confirmed with invalid delivery date

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_confirmed_delivery_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_confirmed_delivery",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_confirmed_delivery": False
    })
)

### Returned with date issue

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_returned_date_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_returned_date",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_returned_date": False
    })
)

### Check All Date Flags

In [0]:
validation_df.select(
    "order_id",
    "order_date",
    "ship_date",
    "delivery_date",
    "order_status",
    "invalid_order_date",
    "invalid_ship_date",
    "invalid_delivery_date",
    "invalid_ship_date_order",
    "invalid_delivery_date_ship",
    "invalid_shipped_date",
    "invalid_deliver_date",
    "invalid_placed_dates",
    "invalid_confirmed_delivery",
    "invalid_returned_date"
).show(truncate=False)
validation_df.count()

+---------+----------+----------+-------------+------------+------------------+-----------------+---------------------+-----------------------+--------------------------+--------------------+--------------------+--------------------+--------------------------+---------------------+
|order_id |order_date|ship_date |delivery_date|order_status|invalid_order_date|invalid_ship_date|invalid_delivery_date|invalid_ship_date_order|invalid_delivery_date_ship|invalid_shipped_date|invalid_deliver_date|invalid_placed_dates|invalid_confirmed_delivery|invalid_returned_date|
+---------+----------+----------+-------------+------------+------------------+-----------------+---------------------+-----------------------+--------------------------+--------------------+--------------------+--------------------+--------------------------+---------------------+
|ORD100218|2025-10-18|2025-10-20|NULL         |Shipped     |false             |false            |false                |false                  |false   

1000

### Step 24.9 — Payment + Order Status business-rule flags.

### Add the Existing Business-Rule Flag

In [0]:
validation_df = (
    validation_df
    .join(
        order_date_analysis
        .select(
            "order_id",
            "payment_order_status_violation"
        )
        .dropDuplicates(['order_id']),
        on="order_id",
        how="left"
    )
    .fillna({
        "payment_order_status_violation":False
    })
)

In [0]:
validation_df.select(
    "order_id",
    "payment_status",
    "order_status",
    "payment_order_status_violation"
).show()

+---------+--------------+------------+------------------------------+
| order_id|payment_status|order_status|payment_order_status_violation|
+---------+--------------+------------+------------------------------+
|ORD100218|          PAID|     Shipped|                         false|
|ORD100164|      REFUNDED|   Cancelled|                         false|
|ORD100064|      REFUNDED|   Cancelled|                         false|
|ORD100596|          PAID|   Delivered|                         false|
|ORD100313|          PAID|   Delivered|                         false|
|ORD100660|          PAID|   Delivered|                         false|
|ORD100071|          PAID|   Delivered|                         false|
|ORD100724|          PAID|    Returned|                         false|
|ORD100851|          PAID|     Shipped|                         false|
|ORD100463|       PENDING|     Shipped|                         false|
|ORD100027|          PAID|   Delivered|                         false|
|ORD10

In [0]:
validation_df.select(
    "order_id",
    "payment_status",
    "order_status",
    "payment_order_status_violation"
).filter(F.col("payment_order_status_violation")==True).show()

+---------+--------------+------------+------------------------------+
| order_id|payment_status|order_status|payment_order_status_violation|
+---------+--------------+------------+------------------------------+
|ORD100896|       PENDING|   Delivered|                          true|
|ORD100846|       PENDING|   Delivered|                          true|
|ORD100812|      REFUNDED|      Placed|                          true|
|ORD100844|      REFUNDED|      Placed|                          true|
|ORD100889|      REFUNDED|      Placed|                          true|
|ORD100843|       PENDING|   Delivered|                          true|
|ORD100853|       PENDING|   Delivered|                          true|
|ORD100790|        FAILED|   Delivered|                          true|
|ORD100890|       PENDING|   Delivered|                          true|
|ORD100877|        FAILED|   Delivered|                          true|
|ORD100776|      REFUNDED|      Placed|                          true|
|ORD10

### Step 24.10 — Shipping City, State and Pincode Validation Flags

### Invalid Shipping City

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_shipping_city_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_shipping_city",F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_shipping_city":False
    })
)

### Invalid Shipping State

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_shipping_state_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_shipping_state",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_shipping_state": False
    })
)

### Invalid Pincode

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_pincode_format_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_pincode_format",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_pincode_format": False
    })
)

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_pincode_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_pincode",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_pincode": False
    })
)

In [0]:
validation_df.select(
    "order_id",
    "shipping_city",
    "shipping_state",
    "pincode",
    "invalid_shipping_city",
    "invalid_shipping_state",
    "invalid_pincode",
    "invalid_pincode_format"
).show(truncate=False)

+---------+-------------+--------------+-------+---------------------+----------------------+---------------+----------------------+
|order_id |shipping_city|shipping_state|pincode|invalid_shipping_city|invalid_shipping_state|invalid_pincode|invalid_pincode_format|
+---------+-------------+--------------+-------+---------------------+----------------------+---------------+----------------------+
|ORD100218|Bengaluru    |Karnataka     |560073 |false                |false                 |false          |false                 |
|ORD100164|Lucknow      |Uttar Pradesh |226094 |false                |false                 |false          |false                 |
|ORD100064|Hyderabad    |Telangana     |500032 |false                |false                 |false          |false                 |
|ORD100596|Pune         |Maharashtra   |411008 |false                |false                 |false          |false                 |
|ORD100313|Madurai      |Tamil Nadu    |625013 |false                

### Step 24.11 — Duplicate and Conflicting Order Flags

### Duplicate Order Flag

In [0]:
validation_df = (
    validation_df
    .join(
        duplicate_orders_ids
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "duplicate_order",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "duplicate_order": False
    })
)

In [0]:
validation_df.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+--------+----------+----------------+------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+--------------+------------+----------------+-----------+---------------------+----------------------+----------------------+----------------+-------------------+------------------+--------------------+----------------+----------------+------------------+------------------------+--------------------+--------------------+----------------------+----------------------+--------------------+------------------+-----------------+---------------------+-----------------------+--------------------------+--------------------+--------------------+--------------------+--------------------------+---------------------+------------------------------+---------------------+--------------------

### Update invalid_customer_name

In [0]:
validation_df = validation_df.withColumn(
    "invalid_customer_name",
    F.when(
        F.col("customer_name").isNull() |
        (F.trim(F.col("customer_name")) == ""),
        F.lit(True)
    )
    .otherwise(F.col("invalid_customer_name"))
)

### update invalid_customer_email

In [0]:
validation_df = validation_df.withColumn(
    "invalid_customer_email",
    F.when(
        F.col("customer_email").isNull() |
        (F.trim(F.col("customer_email")) == ""),
        F.lit(True)
    )
    .otherwise(F.col("invalid_customer_email"))
)

### update invalid_customer_phone

In [0]:
validation_df = validation_df.withColumn(
    "invalid_customer_phone",
    F.when(
        F.col("customer_phone").isNull() |
        (F.trim(F.col("customer_phone")) == ""),
        F.lit(True)
    )
    .otherwise(F.col("invalid_customer_phone"))
)

### update invalid_product_name

In [0]:
validation_df = validation_df.withColumn(
    "invalid_product_name",
    F.when(
        F.col("product_name").isNull() |
        (F.trim(F.col("product_name")) == ""),
        F.lit(True)
    )
    .otherwise(F.col("invalid_product_name"))
)

### Step 25 — Create reject_reason

In [0]:
validation_df = validation_df.withColumn(
    "reject_reason",
    F.concat_ws(
        ", ",
        F.when(
            F.col("invalid_customer_name"),
            F.lit("INVALID_CUSTOMER_NAME")
        ),
        F.when(
            F.col("invalid_customer_email"),
            F.lit("INVALID_CUSTOMER_EMAIL")
        ),
        F.when(
            F.col("invalid_customer_phone"),
            F.lit("INVALID_CUSTOMER_PHONE")
        ),
        F.when(
            F.col("invalid_order_id"),
            F.lit("INVALID_ORDER_ID")
        ),
        F.when(
            F.col("invalid_customer_id"),
            F.lit("INVALID_CUSTOMER_ID")
        ),
        F.when(
            F.col("invalid_product_name"),
            F.lit("INVALID_PRODUCT_NAME")
        ),
        F.when(
            F.col("invalid_category"),
            F.lit("INVALID_CATEGORY")
        ),
        F.when(
            F.col("invalid_quantity"),
            F.lit("INVALIID_QUANTITY")
        ),
        F.when(
            F.col("invalid_unit_price"),
            F.lit("INVALID_UNIT_PRICE")
        ),
        F.when(
            F.col("invalid_discount_percent"),
            F.lit("INVALID_DISCOUNT_PERCENT")
        ),
        F.when(
            F.col("invalid_order_amount"),
            F.lit("INVALID_ORDER_AMOUNT")
        ),
        F.when(
            F.col("invalid_final_amount"),
            F.lit("INVALID_FINAL_AMOUNT")
        ),
        F.when(
            F.col("invalid_payment_method"),
            F.lit("INVALID_PAYMENT_METHOD")
        ),
        F.when(
            F.col("invalid_payment_status"),
            F.lit("INVALID_PAYMENT_STATUS")
        ),
        F.when(
            F.col("invalid_order_status"),
            F.lit("INVALID_ORDER_STATUS")
        ),
        F.when(
            F.col("payment_order_status_violation"),
            F.lit("PAYMENT_ORDER_STATUS_VIOLATION")
        ),
        F.when(
            F.col("invalid_order_date"),
            F.lit("INVALID_ORDER_DATE")
        ),
        F.when(
            F.col("invalid_ship_date"),
            F.lit("INVALID_SHIP_DATE")
        ),
        F.when(
            F.col("invalid_deliver_date"),
            F.lit("INVALID_DELIVERY_DATE")
        ),
        F.when(
            F.col("invalid_ship_date_order"),
            F.lit("SHIP_DATE_BEFORE_ORDER_DATE")
        ),
        F.when(
            F.col("invalid_delivery_date_ship"),
            F.lit("DELIVERY_DATE_BEFORE_SHIP_DATE")
        ),
        F.when(
            F.col("invalid_ship_date"),
            F.lit("SHIPPED_WITHOUT_SHIP_DATE")
        ),
        F.when(
            F.col("invalid_delivery_date"),
            F.lit("DELIVERY_WITHOUT_DELIVERY_DATE")
        ),
        F.when(
            F.col("invalid_placed_dates"),
            F.lit("PLACED_WITH_SHIPPING_OR_DELIVERY_DATE")
        ),
        F.when(
            F.col("invalid_confirmed_delivery"),
            F.lit("CONFIRMED_WITH_INVALID_DELIVERY_DATE")
        ),
        F.when(
            F.col("invalid_returned_date"),
            F.lit("RETURNED_WITH_DATE_ISSUE")
        ),
        F.when(
            F.col("invalid_shipping_city"),
            F.lit("INVALID_SHIPPING_CITY")
        ),
        F.when(
            F.col("invalid_shipping_state"),
            F.lit("INVALID_SHIPPING_STATE")
        ),
        F.when(
            F.col("invalid_pincode"),
            F.lit("INVALID_PINCODE")
        ),
        F.when(
            F.col("invalid_pincode_format"),
            F.lit("INVALID_PINCODE_FORMAT")
        ),
        F.when(
            F.col("duplicate_order"),
            F.lit("DUPLICATE_ORDER")
        ),
    )
)

In [0]:
validation_df.filter(
    F.col("reject_reason")!=""
).show()

+---------+-----------+------------------+--------------------+--------------+----------+-------------------+--------+----------+----------------+------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+--------------+------------+----------------+-----------+---------------------+----------------------+----------------------+----------------+-------------------+------------------+--------------------+----------------+----------------+------------------+------------------------+--------------------+--------------------+----------------------+----------------------+--------------------+------------------+-----------------+---------------------+-----------------------+--------------------------+--------------------+--------------------+--------------------+--------------------------+---------------------+------------------------------+---------------------+-------------------

### Step 26 — Create the Final record_valid Flag

In [0]:
validation_df = validation_df.withColumn(
    "record_valid",
    F.when(
        F.col("reject_reason")=="",
        F.lit(True)
    ).otherwise(F.lit(False))
)

In [0]:
validation_df.select(
    "order_id",
    "reject_reason",
    "record_valid"
).show(truncate=False)

+---------+-----------------------------------------------------------------------------------------+------------+
|order_id |reject_reason                                                                            |record_valid|
+---------+-----------------------------------------------------------------------------------------+------------+
|ORD100218|                                                                                         |true        |
|ORD100164|                                                                                         |true        |
|ORD100064|                                                                                         |true        |
|ORD100596|                                                                                         |true        |
|ORD100313|                                                                                         |true        |
|ORD100660|                                                                     

### Verify the overall counts

In [0]:
validation_df.groupBy("record_valid").count().show()

+------------+-----+
|record_valid|count|
+------------+-----+
|        true|  725|
|       false|  275|
+------------+-----+



### Step 27 — Split Valid and Invalid Records

### Silver candidate records

In [0]:
silver_candiate_df = validation_df.filter(
    F.col("record_valid")==True
    )

### Reject records

In [0]:
reject_df = validation_df.filter(
    F.col("record_valid") == False
)

In [0]:
print("Silver Candiated:",silver_candiate_df.count())
print("Reject records:",reject_df.count())

Silver Candiated: 725
Reject records: 275


In [0]:
print("Bronze:", bronze_orders.count())
print("Validation:", validation_df.count())
print("Silver:", silver_candiate_df.count())
print("Reject:", reject_df.count())

Bronze: 1000
Validation: 1000
Silver: 725
Reject: 275


In [0]:
print(
    "Unique order IDs:",
    validation_df.select("order_id").distinct().count()
)

Unique order IDs: 950


### Step 28 — Write Silver and Reject Data

### Write the Silver Data

In [0]:
silver_candiate_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce_project.silver_orders")

### Write the Reject Data

In [0]:
reject_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce_project.reject_orders")

In [0]:
silver_count=spark.read.table("ecommerce_project.silver_orders").count()
reject_count=spark.read.table("ecommerce_project.reject_orders").count()
print("Silver count:",silver_count)
print("Reject count:",reject_count)

Silver count: 725
Reject count: 275


In [0]:
bronze_count=spark.table("ecommerce_project.bronze_orders").count()

assert silver_count+reject_count == bronze_count

print("Bronze records :",bronze_count)
print("Silver records :",silver_count)
print("Reject records :",reject_count)
print("Record count validation :PASS")

Bronze records : 1000
Silver records : 725
Reject records : 275
Record count validation :PASS


In [0]:
silver_null_summary = silver_candiate_df.select(
    *[
        F.sum(F.col(column).isNull().cast("int")).alias(column)
        for column in silver_candiate_df.columns
    ]
)

In [0]:
silver_null_summary.show()

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+----------+----------------+------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+--------------+------------+--------------+--------+---------------------+----------------------+----------------------+----------------+-------------------+------------------+--------------------+----------------+----------------+------------------+------------------------+--------------------+--------------------+----------------------+----------------------+--------------------+------------------+-----------------+---------------------+-----------------------+--------------------------+--------------------+--------------------+--------------------+--------------------------+---------------------+------------------------------+---------------------+----------------------+----------------------+---

In [0]:
%sql
select * from ecommerce_project.silver_orders;

order_id,customer_id,customer_name,customer_email,customer_phone,product_id,product_name,quantity,unit_price,discount_percent,order_amount,final_amount,order_date,ship_date,delivery_date,shipping_city,shipping_state,pincode,ingestion_timestamp,source_file,batch_id,source_system,pipeline_id,payment_status,order_status,payment_method,category,invalid_customer_name,invalid_customer_email,invalid_customer_phone,invalid_order_id,invalid_customer_id,invalid_product_id,invalid_product_name,invalid_category,invalid_quantity,invalid_unit_price,invalid_discount_percent,invalid_order_amount,invalid_final_amount,invalid_payment_method,invalid_payment_status,invalid_order_status,invalid_order_date,invalid_ship_date,invalid_delivery_date,invalid_ship_date_order,invalid_delivery_date_ship,invalid_shipped_date,invalid_deliver_date,invalid_placed_dates,invalid_confirmed_delivery,invalid_returned_date,payment_order_status_violation,invalid_shipping_city,invalid_shipping_state,invalid_pincode_format,invalid_pincode,duplicate_order,reject_reason,record_valid
ORD100218,CUST10020,Zara Sundaram,zara.sundaram@yahoo.com,8249926919,PROD1064,Cricket Bat,8,5037.33,0,40298.64,40298.64,2025-10-18,2025-10-20,null,Bengaluru,Karnataka,560073,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,PAID,Shipped,NET BANKING,SPORTS,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100164,CUST10042,Mahesh Shetty,mahesh.shetty@rediffmail.com,7678998028,PROD1073,Mouse,4,2091.76,80,8367.04,1673.41,2025-12-13,null,null,Lucknow,Uttar Pradesh,226094,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,REFUNDED,Cancelled,NET BANKING,ELECTRONICS,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100064,CUST10136,Chitra Reddy,chitra.reddy@hotmail.com,8113585139,PROD1066,Books,4,2731.88,10,10927.52,9834.77,2025-11-27,null,null,Hyderabad,Telangana,500032,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,REFUNDED,Cancelled,DEBIT CARD,BOOKS,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100596,CUST10067,Kabir Siddiqui,kabir.siddiqui@gmail.com,8812308209,PROD1036,Water Bottle,9,334.78,80,3013.02,602.6,2025-09-06,2025-09-07,2025-09-08,Pune,Maharashtra,411008,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,PAID,Delivered,UPI,HOME,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100313,CUST10123,Karthik Murthy,karthik.murthy@rediffmail.com,8327268503,PROD1075,Cotton Shirt,10,2379.61,50,23796.1,11898.05,2025-12-12,2025-12-15,2025-12-19,Madurai,Tamil Nadu,625013,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,PAID,Delivered,NET BANKING,FASHION,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100660,CUST10016,Radha Chandran,radha.chandran@rediffmail.com,9389854268,PROD1048,Bluetooth Speaker,5,7784.98,10,38924.9,35032.41,2025-01-25,2025-01-26,2025-01-30,Jaipur,Rajasthan,302000,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,PAID,Delivered,UPI,ELECTRONICS,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100071,CUST10105,Rajesh Siddiqui,rajesh.siddiqui@rediffmail.com,6698509918,PROD1100,Mixer Grinder,6